# RL-05 : MDP, Programmation Dynamique et Q-Learning Tabulaire (C#)

Ce carnet est le jumeau C# de [RL-05-MDP-Programmation-Dynamique-Q-Learning-Python.ipynb](RL-05-MDP-Programmation-Dynamique-Q-Learning-Python.ipynb). Il couvre le meme enchainement, avec les memes experiences et les memes hyperparametres : MDP, Iteration de Valeur, Iteration de Politique, Q-Learning tabulaire, CliffWalking, SARSA, Expected SARSA, n-step TD.

**Aucune dependance externe.** Le BCL .NET 9 suffit : deux grilles sont decrites a la main (`FrozenLake` 4x4 et `CliffWalking` 4x12), leurs transitions sont construites en memoire, les algorithmes sont ecrits a la main, et les figures sont des SVG produits par le carnet. La source Python utilise `gymnasium`, `numpy` et `matplotlib` ; le portage remplace ces trois dependances par des tableaux du BCL et une generation SVG de quelques lignes.

## Objectifs d'apprentissage

A l'issue de ce carnet, vous saurez :

1. **Formaliser un MDP** $(S, A, P, R, \gamma)$ et lire une table de transitions.
2. **Resoudre un MDP connu** par Iteration de Valeur et par Iteration de Politique, et verifier que les deux trouvent le meme optimum.
3. **Implementer le Q-Learning tabulaire** sans jamais ouvrir le modele de transition, puis comparer sa politique a l'optimum calcule.
4. **Mesurer l'effet des hyperparametres** $\alpha$ et $\gamma$ par balayage a graine fixe.
5. **Distinguer off-policy et on-policy** sur CliffWalking, ou la cible de mise a jour change la politique apprise (Q-Learning, SARSA, Expected SARSA).
6. **Placer une methode sur le continuum biais-variance** avec n-step TD.

## Prerequis

- Aucun paquet NuGet, aucun service. Le noyau `.net-csharp` execute du C# compile par .NET Interactive.
- Savoir lire une grille indexee `etat = ligne * colonnes + colonne`.
- Les sections 3 et 4 supposent l'equation d'optimalite de Bellman ; la section 5 en donne l'usage sans modele.


## Références

> *Ancres savantes -- Bellman, R. (1957), Dynamic Programming, Princeton University Press (equation de Bellman, itération de valeur et de politique resolvent le MDP en point fixe) ; Watkins, C.J.C.H. (1989), Learning from Delayed Rewards, These de doctorat, Universite de Cambridge (Q-Learning tabulaire, méthode off-policy model-free qui converge vers Q* sans connaitre le modèle de transition) ; Sutton, R.S. & Barto, A.G. (2018), Reinforcement Learning: An Introduction (2nd ed.), MIT Press (cadre formel du MDP $(S, A, P, R, \gamma)$, value/policy itération, compromis exploration/exploitation).*

Les environnements sont ceux de Gymnasium, decrits ici a la main :

- [FrozenLake](https://gymnasium.farama.org/environments/toy_text/frozen_lake/) — grille 4x4, 16 etats, 4 actions, convention `0 = Gauche, 1 = Bas, 2 = Droite, 3 = Haut`.
- [CliffWalking](https://gymnasium.farama.org/environments/toy_text/cliff_walking/) — grille 4x12, 48 etats, 4 actions, convention `0 = Haut, 1 = Droite, 2 = Bas, 3 = Gauche`.

Les deux conventions d'action diffèrent : c'est un detail de Gymnasium, pas du probleme. Chaque affichage de politique rappelle celle qui s'applique.


## 1. Préparation du noyau

Trois choses se posent une fois pour tout le carnet :

1. **La culture invariante.** Sans elle, un `double` s'afficherait avec une virgule decimale sur une machine francaise et un point ailleurs : les sorties commitees changeraient selon le poste. Les nombres de ce carnet sont donc ecrits avec la culture invariante, et les arrondis sont faits par `Math.Round` avant affichage.
2. **Un type de figure.** `PlotSvg` porte un fragment SVG ; un `Formatter.Register` explique au noyau que l'objet doit etre rendu comme du HTML. Sans cet enregistrement, la sortie afficherait le texte brut du SVG.
3. **Les constantes partagees** (graine, $\alpha$, $\gamma$), pour que toutes les sections parlent du meme protocole.


In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text;
using Microsoft.DotNet.Interactive;
using Microsoft.DotNet.Interactive.Formatting;

// La culture invariante garantit que 0.99 s'ecrit "0.99" et non "0,99",
// et que les sorties sont identiques sur toute machine.
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;

// Un graphe SVG est rendu comme du HTML : la cellule affiche l'image, pas son source.
record PlotSvg(string Content);
string SvgText(string value) => System.Net.WebUtility.HtmlEncode(value);
Formatter.Register<PlotSvg>((value, writer) => writer.Write(value.Content), "text/html");

// Protocole partage par tout le carnet : une graine, un gamma, un alpha.
const int Seed = 42;
const double Gamma = 0.99;
const double Alpha = 0.1;

string cultureName = string.IsNullOrEmpty(CultureInfo.CurrentCulture.Name) ? "invariante" : CultureInfo.CurrentCulture.Name;
Console.WriteLine($"Runtime .NET {Environment.Version} ; culture {cultureName} ; aucune dependance NuGet.");
Console.WriteLine($"Protocole partage : graine {Seed}, gamma {Gamma}, alpha {Alpha}.");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Runtime .NET 10.0.15 ; culture invariante ; aucune dependance NuGet.


Protocole partage : graine 42, gamma 0.99, alpha 0.1.


## 2. Processus de Décision Markovien (MDP)

Un MDP formalise l'interaction agent-environnement par le quintuplet $(S, A, P, R, \gamma)$ : etats, actions, transitions $P(s'|s,a)$, recompenses, facteur d'actualisation. La propriete de Markov stipule que le futur ne depend que de l'etat present, pas de l'historique.

Avant d'ecrire le moindre algorithme, il faut **le modele** : pour chaque couple $(s, a)$, la liste des issues possibles `(probabilite, etat_suivant, recompense, terminal)`. La source Python les lit sur `env.unwrapped.P` ; ici la grille est decrite et les transitions sont construites a la main, ce qui rend le modele lisible au lieu d'etre suppose.

Deux environnements serviront, et ils ne sont pas interchangeables :

- **FrozenLake** est le banc *model-based* : sa table $P$ complete est le materiau des Iterations de Valeur et de Politique.
- **CliffWalking**, en section 6, est le banc *model-free* : les algorithmes n'y ouvriront jamais $P$, ils apprendront de quintuplets observes.

Les structures de recompense s'opposent aussi : FrozenLake est *sparse* (+1 au but, 0 partout ailleurs — 15 cases muettes sur 16), CliffWalking est *dense et punitif* (-1 par pas, -100 pour une chute). Une methode qui ne marcherait que sur l'une des deux ne meriterait pas le nom de generale.


In [2]:
record Transition(double Prob, int NextState, double Reward, bool Done);

sealed class GridEnvironment
{
    public string Name { get; init; } = "";
    public int Rows { get; init; }
    public int Cols { get; init; }
    public char[,] Desc { get; init; } = new char[0, 0];
    public string[] ActionNames { get; init; } = Array.Empty<string>();
    public (int Dr, int Dc)[] Moves { get; init; } = Array.Empty<(int, int)>();
    public List<Transition>[,] P { get; init; } = new List<Transition>[0, 0];
    public int StartState { get; init; }

    public int NStates => Rows * Cols;
    public int NActions => ActionNames.Length;
    public char Cell(int state) => Desc[state / Cols, state % Cols];

    // Echantillonne une issue de P[s, a] : c'est le seul point ou le modele
    // est consulte par un agent en ligne.
    public (int NextState, double Reward, bool Done) Step(int state, int action, Random rng)
    {
        var outcomes = P[state, action];
        if (outcomes.Count == 1)
            return (outcomes[0].NextState, outcomes[0].Reward, outcomes[0].Done);

        double u = rng.NextDouble(), acc = 0.0;
        foreach (var t in outcomes)
        {
            acc += t.Prob;
            if (u < acc) return (t.NextState, t.Reward, t.Done);
        }
        var last = outcomes[^1];
        return (last.NextState, last.Reward, last.Done);
    }
}

// FrozenLake 4x4 : la meme grille que Gymnasium, et les memes transitions.
// Convention d'action : 0 = Gauche, 1 = Bas, 2 = Droite, 3 = Haut.
GridEnvironment MakeFrozenLake(bool slippery = false)
{
    string[] rows = { "SFFF", "FHFH", "FFFH", "HFFG" };
    var desc = new char[4, 4];
    for (int r = 0; r < 4; r++)
        for (int c = 0; c < 4; c++)
            desc[r, c] = rows[r][c];

    var moves = new[] { (0, -1), (1, 0), (0, 1), (-1, 0) };
    var env = new GridEnvironment
    {
        Name = slippery ? "FrozenLake-v1 (is_slippery=True)" : "FrozenLake-v1 (is_slippery=False)",
        Rows = 4, Cols = 4, Desc = desc,
        ActionNames = new[] { "Gauche", "Bas", "Droite", "Haut" },
        Moves = moves,
        P = new List<Transition>[16, 4],
        StartState = 0,
    };

    for (int s = 0; s < 16; s++)
    {
        for (int a = 0; a < 4; a++)
        {
            // Deterministe : l'action voulue, une seule issue.
            // Glissant : l'action voulue et ses deux perpendiculaires, 1/3 chacune.
            int[] directions = slippery ? new[] { a, (a + 1) % 4, (a + 3) % 4 } : new[] { a };
            double prob = 1.0 / directions.Length;
            var list = new List<Transition>(directions.Length);
            foreach (int dir in directions)
            {
                int r = s / 4 + moves[dir].Item1;
                int c = s % 4 + moves[dir].Item2;
                if (r < 0 || r >= 4 || c < 0 || c >= 4) { r = s / 4; c = s % 4; }
                char cell = desc[r, c];
                list.Add(new Transition(prob, r * 4 + c, cell == 'G' ? 1.0 : 0.0, cell == 'H' || cell == 'G'));
            }
            env.P[s, a] = list;
        }
    }
    return env;
}

var env = MakeFrozenLake(slippery: false);
Console.WriteLine($"Espace d'etats   : Discrete({env.NStates})");
Console.WriteLine($"Espace d'actions : Discrete({env.NActions})");
Console.WriteLine();
Console.WriteLine("Grille (S = depart, F = glace, H = trou, G = but), lue sur le modele :");
for (int r = 0; r < env.Rows; r++)
{
    var line = new StringBuilder("  ");
    for (int c = 0; c < env.Cols; c++) line.Append(env.Desc[r, c]).Append(' ');
    Console.WriteLine(line.ToString());
}


Espace d'etats   : Discrete(16)


Espace d'actions : Discrete(4)


Grille (S = depart, F = glace, H = trou, G = but), lue sur le modele :


  S F F F 


  F H F H 


  F F F H 


  H F F G 


**Interroger le modèle.** La classe ci-dessus construit `P[s, a]` : la liste des issues possibles d'une action dans un état, chacune avec sa probabilité. Avant de résoudre quoi que ce soit, on lit ces transitions à la main — c'est le seul endroit du carnet où `P` est consulté directement. Les agents des sections 5 à 9 n'y auront plus accès : ils n'échantillonneront `Step` qu'au fil de leurs épisodes, comme un agent réel le ferait.

In [3]:
string FormatOutcomes(int state, int action)
{
    var parts = env.P[state, action]
        .Select(t => $"({t.Prob:0.00}, {t.NextState}, {t.Reward:0.#}, {(t.Done ? "True" : "False")})");
    return "[" + string.Join(", ", parts) + "]";
}

Console.WriteLine($"P[0][2] (etat 0, Droite) = {FormatOutcomes(0, 2)}");
Console.WriteLine($"P[1][1] (etat 1, Bas)    = {FormatOutcomes(1, 1)}");
Console.WriteLine();
Console.WriteLine($"Case 0 : '{env.Cell(0)}' ; case 1 : '{env.Cell(1)}' ; case 5 : '{env.Cell(5)}'.");
Console.WriteLine($"Le but est l'etat {env.NStates - 1} ('{env.Cell(env.NStates - 1)}'), le depart l'etat {env.StartState}.");


P[0][2] (etat 0, Droite) = [(1.00, 1, 0, False)]


P[1][1] (etat 1, Bas)    = [(1.00, 5, 0, True)]


Case 0 : 'S' ; case 1 : 'F' ; case 5 : 'H'.


Le but est l'etat 15 ('G'), le depart l'etat 0.


### Deux formateurs de lecture

Un tableau de nombres se lit à l'œil nu ; une figure sert à voir la **forme** d'une politique. Les deux formateurs ci-dessous restent donc en texte et seront réutilisés tels quels par les sections suivantes — la table des valeurs pour l'itération de valeur, la grille de flèches pour toute politique, qu'elle soit optimale ou apprise.

Les états terminaux (`H`, `G`) ne portent pas de flèche : on n'en sort pas, et la politique n'y est pas définie.

In [4]:
// Une table de valeurs lue comme la grille : une colonne par case, alignee sur la description.
string RenderDoubleGrid(GridEnvironment e, double[] values)
{
    var sb = new StringBuilder();
    for (int r = 0; r < e.Rows; r++)
    {
        for (int c = 0; c < e.Cols; c++)
            sb.Append(c == 0 ? $"{values[r * e.Cols + c],11:0.000}" : $"{values[r * e.Cols + c],8:0.000}");
        if (r < e.Rows - 1) sb.AppendLine();
    }
    return sb.ToString();
}

// La politique en fleches. arrows est indexe par action, dans l'ordre de l'environnement
// (0 = Gauche, 1 = Bas, 2 = Droite, 3 = Haut) : passer le bon tableau, sinon la grille ment.
string RenderPolicy(GridEnvironment e, int[] policy, string[] arrows)
{
    var sb = new StringBuilder();
    for (int r = 0; r < e.Rows; r++)
    {
        for (int c = 0; c < e.Cols; c++)
        {
            int s = r * e.Cols + c;
            char cell = e.Cell(s);
            string label = (cell == 'H' || cell == 'G') ? cell.ToString() : arrows[policy[s]];
            sb.Append(c == 0 ? $"{label,11}" : $"{label,8}");
        }
        if (r < e.Rows - 1) sb.AppendLine();
    }
    return sb.ToString();
}

**Clé de lecture de la grille** : `S` = départ (*start*), `F` = glace libre (*frozen*, aucune conséquence), `H` = trou (*hole*, l'épisode s'y termine), `G` = but (*goal*, récompense 1). La même clé sert aux couleurs de `VisualizePolicy` plus bas : `S` vert, `F` bleu clair, `H` rouge, `G` doré. Les flèches y sont les actions greedy, dans la convention numérique `0` = Gauche, `1` = Bas, `2` = Droite, `3` = Haut.

Chaque transition est un quadruplet `(probabilite, etat_suivant, recompense, est_terminal)`. En mode non-glissant, chaque action a une probabilite de `1.00` de reussir.

Les deux exemples affiches meritent une lecture lente. Depuis l'etat 0 (le depart `S`), action Droite : `(1.00, 1, 0, False)` — probabilite 1, etat 1, recompense 0, non terminal : un deplacement deterministe banal. Depuis l'etat 1, action Bas : `(1.00, 5, 0, True)` — l'etat 5 est un **trou** (deuxieme ligne de la grille, case `H`), et le drapeau `True` dit que l'episode s'y termine. Un seul mauvais pas depuis la case voisine du depart tue l'episode avec une recompense nulle : c'est ce qui rend certaines actions interdites en pratique et donne son relief a la fonction de valeur. En mode glissant (`is_slippery=True`, l'exercice 1), chacune de ces listes singleton deviendrait une liste de trois issues possibles et la certitude disparaitrait.


## 3. Itération de Valeur (Value Itération)

L'Itération de Valeur resout le MDP en appliquant iterativement l'equation d'optimalite de Bellman :

$$V(s) = \max_a \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V(s') \right]$$

L'algorithme met a jour $V(s)$ pour chaque etat jusqu'a convergence. La politique optimale est ensuite obtenue en choisissant l'action qui maximise $V$ dans chaque etat.


In [5]:
(double[] V, int[] Policy, List<double> Deltas) ValueIteration(
    GridEnvironment env, double gamma = 0.99, double theta = 1e-8, int maxIterations = 1000)
{
    int nS = env.NStates, nA = env.NActions;
    var V = new double[nS];
    var deltas = new List<double>();

    for (int iteration = 0; iteration < maxIterations; iteration++)
    {
        double delta = 0;
        for (int s = 0; s < nS; s++)
        {
            double v = V[s];
            double best = double.NegativeInfinity;
            for (int a = 0; a < nA; a++)
            {
                double q = 0;
                foreach (var t in env.P[s, a])
                    q += t.Prob * (t.Reward + gamma * V[t.NextState] * (t.Done ? 0 : 1));
                if (q > best) best = q;
            }
            V[s] = best;
            delta = Math.Max(delta, Math.Abs(v - V[s]));
        }
        deltas.Add(delta);
        if (delta < theta)
        {
            Console.WriteLine($"Convergence a l'iteration {iteration + 1} (delta = {delta:0.00e+00}).");
            break;
        }
    }

    var policy = new int[nS];
    for (int s = 0; s < nS; s++)
    {
        double best = double.NegativeInfinity;
        for (int a = 0; a < nA; a++)
        {
            double q = 0;
            foreach (var t in env.P[s, a])
                q += t.Prob * (t.Reward + gamma * V[t.NextState] * (t.Done ? 0 : 1));
            if (q > best) { best = q; policy[s] = a; }
        }
    }
    return (V, policy, deltas);
}

In [6]:
var frozen = MakeFrozenLake(slippery: false);
var (vStar, piStar, deltasVi) = ValueIteration(frozen, gamma: Gamma);

Console.WriteLine($"V* — {frozen.Name}, gamma = {Gamma} :");
Console.WriteLine(RenderDoubleGrid(frozen, vStar));
Console.WriteLine();
Console.WriteLine("Politique optimale (fleches = action gloutonne, H/G = etat terminal) :");
Console.WriteLine(RenderPolicy(frozen, piStar, new[] { "<", "v", ">", "^" }));
Console.WriteLine($"V*(depart) = {vStar[frozen.StartState]:0.0000}  |  iterations tracees = {deltasVi.Count}");
Console.WriteLine($"gamma^5 = {Math.Pow(Gamma, 5):0.0000} — le chemin optimal depuis S fait 6 pas, donc V*(0) = gamma^5.");

// Le meme algorithme, sur le modele stochastique : trois issues par action.
var frozenSlippery = MakeFrozenLake(slippery: true);
var (vStarSlip, piStarSlip, deltasSlip) = ValueIteration(frozenSlippery, gamma: Gamma);

Console.WriteLine();
Console.WriteLine($"V* — {frozenSlippery.Name}, gamma = {Gamma} :");
Console.WriteLine(RenderDoubleGrid(frozenSlippery, vStarSlip));
Console.WriteLine($"V*(depart) = {vStarSlip[frozenSlippery.StartState]:0.0000}  |  iterations tracees = {deltasSlip.Count}");
Console.WriteLine($"Delta final du cas glissant : {deltasSlip[^1]:0.00e+00} (borne theta = 1e-8).");

Convergence a l'iteration 7 (delta = 0.00e+00).


V* — FrozenLake-v1 (is_slippery=False), gamma = 0.99 :


      0.951   0.961   0.970   0.961
      0.961   0.970   0.980   0.970
      0.970   0.980   0.990   1.000
      0.980   0.990   1.000   1.000


Politique optimale (fleches = action gloutonne, H/G = etat terminal) :


          v       >       v       <
          v       H       v       H
          >       v       v       H
          H       >       >       G


V*(depart) = 0.9510  |  iterations tracees = 7


... (2 outputs left out)

V* — FrozenLake-v1 (is_slippery=True), gamma = 0.99 :


      0.542   0.499   0.471   0.457
      0.558   0.561   0.358   0.269
      0.592   0.643   0.615   0.536
      0.440   0.742   0.863   0.951


V*(depart) = 0.5420  |  iterations tracees = 324


Delta final du cas glissant : 9.69e-09 (borne theta = 1e-8).


### Lecture du résultat

Les deux modèles tournent sur **le même code**, et le contraste est la leçon :

| Modèle | Balayages | `delta` final | $V^*(S)$ |
|---|---|---|---|
| non glissant | **7** | exactement `0` | **0.951** |
| glissant | **324** | `9.69e-09` (< `theta = 1e-8`) | **0.542** |

En mode déterministe, la propagation remonte le long du chemin sans aucune atténuation numérique : l'écart maximal tombe **exactement à zéro** au septième balayage, au lieu de décroître asymptotiquement. Sur le modèle stochastique il faut 46 fois plus de balayages, et `delta` s'arrête seulement parce que `theta = 1e-8` l'autorise — la décroissance est géométrique, de raison proche de `gamma`, donc l'échelle logarithmique de la courbe suivante y apparaît comme une droite.

Quatre valeurs du modèle déterministe se lisent directement dans la table et servent de contrôle. La colonne « pas » compte les déplacements jusqu'à la récompense, et $V^*(s) = \gamma^{\text{pas}-1}$ :

| État | $V^*$ | Pas jusqu'au but | $\gamma^{\text{pas}-1}$ |
|---|---|---|---|
| 0 (`S`) | **0.951** | 6 | $0.99^5 = 0.951$ |
| 4 | **0.961** | 5 | $0.99^4 = 0.961$ |
| 9 | **0.980** | 3 | $0.99^2 = 0.980$ |
| 14 | **1.000** | 1 | $0.99^0 = 1.000$ |

L'état 15 (`G`) vaut **1.000** aussi : la récompense y est perçue sans délai, donc sans actualisation. Chaque pas supplémentaire multiplie la valeur par `gamma = 0.99` — la table n'est rien d'autre que la distance au but, lue à travers le facteur d'actualisation. C'est le résultat attendu du chemin le plus court, et c'est aussi la preuve que la reconstruction manuelle de $P$ est fidèle : une seule transition mal orientée, et l'une de ces quatre valeurs serait fausse.

Le modèle glissant le confirme par un second chemin. Sa table n'est plus lisible à la main, mais \(V^*(0) = 0.542\) est la valeur publiée de `FrozenLake-v1` 4x4 avec `is_slippery=True` et \(\gamma = 0.99\). L'obtenir sans importer Gymnasium, avec trois issues à \(1/3\) par action, c'est reproduire la sémantique exacte de l'environnement — y compris le détail qui piège le plus souvent : la glissance s'applique aussi au dernier pas, donc le but peut être manqué depuis la case voisine.

Un point surprend souvent : les **trous** (`H`, états 5, 7, 11, 12) portent eux aussi une valeur non nulle. C'est correct et non un bug. Le drapeau `terminal` de Gymnasium n'est vrai que sur la transition qui **arrive** dans un trou ; depuis un trou, les actions repartent vers les cases voisines avec une récompense nulle. $V^*$ d'un état terminal n'a donc pas de sens physique — il n'est jamais consulté par un agent, qui s'arrête en y entrant — mais il prend la valeur de ce qu'on pourrait encore y gagner. La politique affichée remplace d'ailleurs le glyphe par `H` / `G` pour ces cases, précisément parce qu'une flèche y serait un mensonge.

La politique optimale est enfin **lisible d'un coup d'œil** : depuis `S`, on descend, puis à droite, puis on longe la colonne du but. Aucune action ne pointe vers un trou — ce qui n'a rien d'automatique, et qu'un balayage trop court aurait laissé inachevé.


### Visualisation de la politique

Une table de flèches se lit, mais une grille colorée se comprend. `DrawGrid` compose du SVG à la main — aucune dépendance graphique, comme pour le reste du carnet — et réutilise le même vocabulaire de couleurs que la clé de lecture : `S` vert, `F` bleu clair, `H` rouge, `G` doré. La flèche de chaque case est l'action gloutonne, et la valeur $V^*$ est rappelée en petit sous le glyphe.

La fonction servira telle quelle pour les politiques apprises en section 5 et pour CliffWalking en section 6 : le même dessin, une autre politique.


In [7]:
PlotSvg DrawGrid(GridEnvironment e, int[] policy, string[] arrows, string title,
                 double[] values = null, string note = "")
{
    double cell = 76, pad = 16;
    double header = 46, footer = string.IsNullOrEmpty(note) ? 10 : 30;
    double w = e.Cols * cell + 2 * pad;
    double h = header + e.Rows * cell + footer;

    var fill = new Dictionary<char, string>
    {
        ['S'] = "#a8e6a3", ['F'] = "#e6f2fb", ['H'] = "#f4a8a8", ['G'] = "#f7d774", ['C'] = "#4a4a4a",
    };

    var sb = new StringBuilder();
    sb.Append($"<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"{w:0}\" height=\"{h:0}\" viewBox=\"0 0 {w:0} {h:0}\" font-family=\"Segoe UI, sans-serif\">");
    sb.Append($"<text x=\"{w / 2:0}\" y=\"26\" text-anchor=\"middle\" font-size=\"15\" font-weight=\"600\" fill=\"#1f2933\">{SvgText(title)}</text>");

    for (int r = 0; r < e.Rows; r++)
    {
        for (int c = 0; c < e.Cols; c++)
        {
            int s = r * e.Cols + c;
            char kind = e.Cell(s);
            double x = pad + c * cell, y = header + r * cell;
            sb.Append($"<rect x=\"{x:0}\" y=\"{y:0}\" width=\"{cell - 4:0}\" height=\"{cell - 4:0}\" rx=\"8\" fill=\"{fill.GetValueOrDefault(kind, "#eeeeee")}\" stroke=\"#9aa5b1\" stroke-width=\"1\"/>");

            bool terminal = kind == 'H' || kind == 'G';
            string glyph = terminal ? kind.ToString() : (policy == null ? "" : arrows[policy[s]]);
            int glyphSize = terminal ? 22 : 28;
            double glyphY = y + (cell - 4) / 2 + (values == null ? 8 : 2);
            sb.Append($"<text x=\"{x + (cell - 4) / 2:0}\" y=\"{glyphY:0}\" text-anchor=\"middle\" font-size=\"{glyphSize}\" fill=\"#1f2933\">{SvgText(glyph)}</text>");

            if (values != null)
                sb.Append($"<text x=\"{x + (cell - 4) / 2:0}\" y=\"{y + cell - 14:0}\" text-anchor=\"middle\" font-size=\"11\" fill=\"#52606d\">{values[s]:0.00}</text>");
        }
    }

    if (!string.IsNullOrEmpty(note))
        sb.Append($"<text x=\"{w / 2:0}\" y=\"{h - 8:0}\" text-anchor=\"middle\" font-size=\"11\" fill=\"#52606d\">{SvgText(note)}</text>");

    sb.Append("</svg>");
    return new PlotSvg(sb.ToString());
}

// Convention d'action de FrozenLake : 0 = Gauche, 1 = Bas, 2 = Droite, 3 = Haut.
var arrows = new[] { "←", "↓", "→", "↑" };

DrawGrid(frozen, piStar, arrows,
         "Politique optimale — FrozenLake 4x4, is_slippery = False",
         vStar,
         $"V* en petit sous chaque fleche ; gamma = {Gamma}, theta = 1e-8, convergence en {deltasVi.Count} balayages").Display();

Politique optimale — FrozenLake 4x4, is_slippery = False ↓ 0.95 → 0.96 ↓ 0.97 ← 0.96 ↓ 0.96 H 0.97 ↓ 0.98 H 0.97 → 0.97 ↓ 0.98 ↓ 0.99 H 1.00 H 0.98 → 0.99 → 1.00 G 1.00 V* en petit sous chaque fleche ; gamma = 0.99, theta = 1e-8, convergence en 7 balayages

### Vitesse de convergence

`ValueIteration` renvoie aussi la trace de `delta`, l'écart maximal entre deux balayages successifs. Cette trace est l'instrument de diagnostic de tout algorithme itératif : tant qu'elle ne tombe pas sous `theta`, la politique affichée n'est pas garantie optimale.

Les deux modèles ne demandent pas la même échelle, et c'est le second enseignement du carnet : **l'échelle se choisit d'après la forme mesurée**, jamais par habitude.

- **Non glissant**, échelle linéaire : `delta` reste collé autour de `0.95` pendant six balayages puis **s'effondre à zéro**. Rien à voir avec une convergence progressive — la valeur ne progresse que lorsque le chemin entier est propagé, et pas un peu à chaque tour. Sur une échelle logarithmique, cette courbe serait illisible (un plat en haut, un point isolé en bas).
- **Glissant**, échelle logarithmique : la décroissance est géométrique, de raison proche de `gamma`. En logarithme elle devient une **droite**, et sa pente se lit directement — c'est la signature d'une contraction de facteur `gamma` à chaque balayage.

Un `delta` exactement nul n'a pas de logarithme : `PlotLines` l'omet du tracé plutôt que de le remplacer par un plancher inventé, et le graphe non glissant est donc en linéaire, où le zéro est représentable.

`PlotLines` est le traceur générique du carnet : plusieurs séries, axes gradués, légende, échelle log optionnelle. Il resservira pour les courbes d'apprentissage et les balayages d'hyperparamètres.


In [8]:
PlotSvg PlotLines(string title, string xLabel, string yLabel,
                  IReadOnlyList<(string Label, double[] Xs, double[] Ys, string Color)> series,
                  bool logY = false)
{
    const double W = 680, H = 380, left = 74, right = 158, top = 48, bottom = 56;
    double pw = W - left - right, ph = H - top - bottom;

    double xMin = double.MaxValue, xMax = double.MinValue;
    double yMin = double.MaxValue, yMax = double.MinValue;
    foreach (var s in series)
    {
        for (int i = 0; i < s.Ys.Length; i++)
        {
            double yv = s.Ys[i];
            // Sur une echelle log, une valeur nulle ou negative n'a pas de place :
            // elle est omise du trace, jamais remplacee par un plancher invente.
            if (logY) { if (yv <= 0) continue; yv = Math.Log10(yv); }
            if (double.IsNaN(yv) || double.IsInfinity(yv)) continue;
            xMin = Math.Min(xMin, s.Xs[i]); xMax = Math.Max(xMax, s.Xs[i]);
            yMin = Math.Min(yMin, yv); yMax = Math.Max(yMax, yv);
        }
    }
    if (yMax - yMin < 1e-12) { yMax += 0.5; yMin -= 0.5; }
    double padY = (yMax - yMin) * 0.08;
    yMin -= padY; yMax += padY;

    double PX(double xv) => left + (xv - xMin) / Math.Max(xMax - xMin, 1e-12) * pw;
    double PY(double yv) => top + ph - (yv - yMin) / (yMax - yMin) * ph;

    var sb = new StringBuilder();
    sb.Append($"<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"{W}\" height=\"{H}\" viewBox=\"0 0 {W} {H}\" font-family=\"Segoe UI, sans-serif\">");
    sb.Append($"<rect x=\"0\" y=\"0\" width=\"{W}\" height=\"{H}\" fill=\"#ffffff\"/>");
    sb.Append($"<text x=\"{left + pw / 2:0}\" y=\"26\" text-anchor=\"middle\" font-size=\"15\" font-weight=\"600\" fill=\"#1f2933\">{SvgText(title)}</text>");

    for (int k = 0; k <= 4; k++)
    {
        double tickY = yMin + (yMax - yMin) * k / 4;
        double pixelY = PY(tickY);
        sb.Append($"<line x1=\"{left}\" y1=\"{pixelY:0.#}\" x2=\"{left + pw:0.#}\" y2=\"{pixelY:0.#}\" stroke=\"#e4e7eb\" stroke-width=\"1\"/>");
        double rawTick = logY ? Math.Pow(10, tickY) : tickY;
        string tickText = logY ? rawTick.ToString("0.###e+0") : rawTick.ToString("0.####");
        sb.Append($"<text x=\"{left - 8}\" y=\"{pixelY + 4:0.#}\" text-anchor=\"end\" font-size=\"11\" fill=\"#52606d\">{SvgText(tickText)}</text>");
    }

    int nTicks = 5;
    for (int k = 0; k <= nTicks; k++)
    {
        double tickX = xMin + (xMax - xMin) * k / nTicks;
        double pixelX = PX(tickX);
        string tickText = tickX.ToString("0.###");
        sb.Append($"<line x1=\"{pixelX:0.#}\" y1=\"{top + ph:0.#}\" x2=\"{pixelX:0.#}\" y2=\"{top + ph + 5:0.#}\" stroke=\"#9aa5b1\" stroke-width=\"1\"/>");
        sb.Append($"<text x=\"{pixelX:0.#}\" y=\"{top + ph + 20:0.#}\" text-anchor=\"middle\" font-size=\"11\" fill=\"#52606d\">{SvgText(tickText)}</text>");
    }

    sb.Append($"<line x1=\"{left}\" y1=\"{top + ph:0.#}\" x2=\"{left + pw:0.#}\" y2=\"{top + ph:0.#}\" stroke=\"#52606d\" stroke-width=\"1.2\"/>");
    sb.Append($"<line x1=\"{left}\" y1=\"{top:0.#}\" x2=\"{left}\" y2=\"{top + ph:0.#}\" stroke=\"#52606d\" stroke-width=\"1.2\"/>");
    sb.Append($"<text x=\"{left + pw / 2:0}\" y=\"{H - 12:0}\" text-anchor=\"middle\" font-size=\"12\" fill=\"#1f2933\">{SvgText(xLabel)}</text>");
    double midY = top + ph / 2;
    sb.Append($"<text x=\"16\" y=\"{midY:0}\" text-anchor=\"middle\" font-size=\"12\" fill=\"#1f2933\" transform=\"rotate(-90 16 {midY:0})\">{SvgText(yLabel)}</text>");

    foreach (var s in series)
    {
        var pts = new StringBuilder();
        for (int i = 0; i < s.Ys.Length; i++)
        {
            double yv = s.Ys[i];
            if (logY) { if (yv <= 0) continue; yv = Math.Log10(yv); }
            if (double.IsNaN(yv) || double.IsInfinity(yv)) continue;
            if (pts.Length > 0) pts.Append(' ');
            pts.Append($"{PX(s.Xs[i]):0.#},{PY(yv):0.#}");
        }
        sb.Append($"<polyline points=\"{pts}\" fill=\"none\" stroke=\"{s.Color}\" stroke-width=\"2.2\" stroke-linejoin=\"round\"/>");
    }

    double legendY = top + 10;
    foreach (var s in series)
    {
        sb.Append($"<line x1=\"{left + pw + 14}\" y1=\"{legendY:0.#}\" x2=\"{left + pw + 38}\" y2=\"{legendY:0.#}\" stroke=\"{s.Color}\" stroke-width=\"2.2\"/>");
        sb.Append($"<text x=\"{left + pw + 44}\" y=\"{legendY + 4:0.#}\" font-size=\"11\" fill=\"#1f2933\">{SvgText(s.Label)}</text>");
        legendY += 18;
    }

    sb.Append("</svg>");
    return new PlotSvg(sb.ToString());
}

double[] Iterations(int n) => Enumerable.Range(1, n).Select(i => (double)i).ToArray();

// Echelle lineaire : le delta deterministe reste colle a 0.95 puis s'effondre a 0.
PlotLines($"Convergence — {frozen.Name} ({deltasVi.Count} balayages)",
          "balayage", "delta (echelle lineaire)",
          new[] { ("delta maximal", Iterations(deltasVi.Count), deltasVi.ToArray(), "#2f6fb3") },
          logY: false).Display();

// Echelle log : la decroissance geometrique du modele glissant devient une droite.
PlotLines($"Convergence — {frozenSlippery.Name} ({deltasSlip.Count} balayages)",
          "balayage", "delta (echelle log)",
          new[] { ("delta maximal", Iterations(deltasSlip.Count), deltasSlip.ToArray(), "#c0504d") },
          logY: true).Display();

Console.WriteLine($"Non glissant : {deltasVi.Count} balayages, delta final = {deltasVi[^1]:0.####e+00} (chute exacte a zero).");
Console.WriteLine($"Glissant     : {deltasSlip.Count} balayages, delta final = {deltasSlip[^1]:0.00e+00}, sous theta = 1e-8.");

Convergence — FrozenLake-v1 (is_slippery=False) (7 balayages) -0.08 0.21 0.5 0.79 1.08 1 2.2 3.4 4.6 5.8 7 balayage delta (echelle lineaire) delta maximal

Convergence — FrozenLake-v1 (is_slippery=True) (324 balayages) 2.261e-9 4.429e-7 8.677e-5 1.7e-2 3.331e+0 1 65.6 130.2 194.8 259.4 324 balayage delta (echelle log) <polyline points="74,67 75.4,88.9 76.8,92.8 78.2,95.6 79.5,99 80.9,102.4 82.3,105.5 83.7,108 85.1,109.8 86.5,111 87.9,112.4 89.3,113.7 90.6,114.5 92,115.1 93.4,115.5 94.8,115.7 96.2,116.1 97.6,116.5 99,117 100.4,117.6 101.7,118.2 103.1,118.8 104.5,119.5 105.9,120.1 107.3,120.8 108.7,121.5 110.1,122.2 111.4,123 112.8,123.7 114.2,124.4 115.6,125.1 117,125.9 118.4,126.6 119.8,127.3 121.2,128.1 122.5,128.8 123.9,129.6 125.3,130.3 126.7,131 128.1,131.8 129.5,132.5 130.9,133 132.3,133.5 133.6,133.7 135,134 136.4,134.3 137.8,134.6 139.2,135 140.6,135.3 142,135.7 143.3,136.2 144.7,136.6 146.1,137.1 147.5,137.5 148.9,138 150.3,138.5 151.7,139 153.1,139.5 154.4,140 155.8,140.6 157.2,141.1 158.6,141.6 160,142.2 161.4,142.7 162.8,143.3 164.2,143.9 165.5,144.4 166.9,145 168.3,145.6 169.7,146.2 171.1,146.8 172.5,147.3 173.9,147.9 175.3,148.5 176.6,149.1 178,149.7 179.4,150.3 180.8,150.9 182.2,151.5 183.6,152.1 185,152.7 186.3,153.4 187.7,154 189.1,154.6 190.5,155.2 191.9,155.8 193.3,156.4 194.7,157 196.1,157.7 197.4,158.3 198.8,158.9 200.2,159.5 201.6,160.1 203,160.7 204.4,161.4 205.8,162 207.2,162.6 208.5,163.2 209.9,163.9 211.3,164.5 212.7,165.1 214.1,165.7 215.5,166.3 216.9,167 218.2,167.6 219.6,168.2 221,168.8 222.4,169.5 223.8,170.1 225.2,170.7 226.6,171.3 228,172 229.3,172.6 230.7,173.2 232.1,173.8 233.5,174.5 234.9,175.1 236.3,175.7 237.7,176.3 239.1,177 240.4,177.6 241.8,178.2 243.2,178.9 244.6,179.5 246,180.1 247.4,180.7 248.8,181.4 250.1,182 251.5,182.6 252.9,183.2 254.3,183.9 255.7,184.5 257.1,185.1 258.5,185.7 259.9,186.4 261.2,187 262.6,187.6 264,188.3 265.4,188.9 266.8,189.5 268.2,190.1 269.6,190.8 271,191.4 272.3,192 273.7,192.6 275.1,193.3 276.5,193.9 277.9,194.5 279.3,195.2 280.7,195.8 282,196.4 283.4,197 284.8,197.7 286.2,198.3 287.6,198.9 289,199.5 290.4,200.2 291.8,200.8 293.1,201.4 294.5,202.1 295.9,202.7 297.3,203.3 298.7,203.9 300.1,204.6 301.5,205.2 302.9,205.8 304.2,206.4 305.6,207.1 307,207.7 308.4,208.3 309.8,209 311.2,209.6 312.6,210.2 314,210.8 315.3,211.5 316.7,212.1 318.1,212.7 319.5,213.4 320.9,214 322.3,214.6 323.7,215.2 325,215.9 326.4,216.5 327.8,217.1 329.2,217.7 330.6,218.4 332,219 333.4,219.6 334.8,220.3 336.1,220.9 337.5,221.5 338.9,222.1 340.3,222.8 341.7,223.4 343.1,224 344.5,224.6 345.9,225.3 347.2,225.9 348.6,226.5 350,227.2 351.4,227.8 352.8,228.4 354.2,229 355.6,229.7 356.9,230.3 358.3,230.9 359.7,231.5 361.1,232.2 362.5,232.8 363.9,233.4 365.3,234.1 366.7,234.7 368,235.3 369.4,235.9 370.8,236.6 372.2,237.2 373.6,237.8 375,238.5 376.4,239.1 377.8,239.7 379.1,240.3 380.5,241 381.9,241.6 383.3,242.2 384.7,242.8 386.1,243.5 387.5,244.1 388.8,244.7 390.2,245.4 391.6,246 393,246.6 394.4,247.2 395.8,247.9 397.2,248.5 398.6,249.1 399.9,249.7 401.3,250.4 402.7,251 404.1,251.6 405.5,252.3 406.9,252.9 408.3,253.5 409.7,254.1 411,254.8 412.4,255.4 413.8,256 415.2,256.6 416.6,257.3 418,257.9 419.4,258.5 420.7,259.2 422.1,259.8 423.5,260.4 424.9,261 426.3,261.7 427.7,262.3 429.1,262.9 430.5,263.6 431.8,264.2 433.2,264.8 434.6,265.4 436,266.1 437.4,266.7 438.8,267.3 440.2,267.9 441.6,268.6 442.9,269.2 444.3,269.8 445.7,270.5 447.1,271.1 448.5,271.7 449.9,272.3 451.3,273 452.7,273.6 454,274.2 455.4,274.8 456.8,275.5 458.2,276.1 459.6,276.7 461,277.4 462.4,278 463.7,278.6 465.1,279.2 466.5,279.9 467.9,280.5 469.3,281.1 470.7,281.7 472.1,282.4 473.5,283 474.8,283.6 476.2,284.3 477.6,284.9 479,285.5 480.4,286.1 481.8,286.8 483.2,287.4 484.6,288 485.9,288.7 487.3,289.3 488.7,289.9 490.1,290.5 491.5,291.2 492.9,291.8 494.3,292.4 495.6,293 497,293.7 498.4,294.3 499.8,294.9 501.2,295.6 502.6,296.2 504,296.8 505.4,297.4 506.7,298.1 508.1,298.7 509.5,299.3 510.9,299.9 512.3,300.6 513.7,301.2 515.1,301.8 516.5,302.5 517.8,303.1 519.2,303.7 520.6,304.3 522,305" fill="none" stroke="#c0504d" stroke-width="2.2" stroke-linejoin="round"/> delta maximal

Non glissant : 7 balayages, delta final = 0e+00 (chute exacte a zero).


Glissant     : 324 balayages, delta final = 9.69e-09, sous theta = 1e-8.


## 4. Itération de Politique (Policy Itération)

L'Itération de Valeur optimisait la **valeur** et laissait la politique suivre. L'Itération de Politique fait l'inverse : elle évalue une politique **fixe** jusqu'à son équilibre, puis l'améliore d'un cran, et recommence.

Le cycle a deux temps, et ils répondent à deux équations différentes :

1. **Évaluation** — résoudre $V^\pi$ pour la politique courante, c'est-à-dire l'équation de Bellman *sans* le $\max$ :
   $$V^\pi(s) = \sum_{s'} P(s'|s,\pi(s)) \left[ R + \gamma V^\pi(s') \right]$$
2. **Amélioration** — remplacer $\pi(s)$ par l'action gloutonne sur $V^\pi$ :
   $$\pi'(s) = \arg\max_a \sum_{s'} P(s'|s,a) \left[ R + \gamma V^\pi(s') \right]$$

Le **théorème d'amélioration de politique** garantit que chaque cycle ne dégrade jamais $V^\pi$ : la séquence est monotone croissante et bornée, donc elle se stabilise en un nombre fini d'itérations. C'est un résultat exact, pas une heuristique de convergence — et c'est pourquoi l'algorithme s'arrête sur un critère **structurel** (la politique ne change plus) plutôt que sur un seuil numérique.

Le prix à payer est l'évaluation : chaque cycle résout un système linéaire, ici par balayages de Gauss-Seidel jusqu'à `theta`. L'algorithme échange donc beaucoup de calcul par cycle contre très peu de cycles.


In [9]:
(double[] V, int[] Policy, List<(int Sweeps, int Changed)> History) PolicyIteration(
    GridEnvironment env, double gamma = 0.99, double theta = 1e-8, int maxIterations = 100)
{
    int nS = env.NStates, nA = env.NActions;
    var policy = new int[nS];            // politique initiale : l'action 0 partout
    var V = new double[nS];
    var history = new List<(int Sweeps, int Changed)>();

    for (int iteration = 0; iteration < maxIterations; iteration++)
    {
        // 1. Evaluation de la politique courante : V^pi, jusqu'a convergence.
        int sweeps = 0;
        while (true)
        {
            sweeps++;
            double delta = 0;
            for (int s = 0; s < nS; s++)
            {
                double previous = V[s];
                double q = 0;
                foreach (var t in env.P[s, policy[s]])
                    q += t.Prob * (t.Reward + gamma * V[t.NextState] * (t.Done ? 0 : 1));
                V[s] = q;
                delta = Math.Max(delta, Math.Abs(previous - V[s]));
            }
            if (delta < theta) break;
        }

        // 2. Amelioration : la politique devient gloutonne sur V^pi.
        int changed = 0;
        for (int s = 0; s < nS; s++)
        {
            int oldAction = policy[s];
            double best = double.NegativeInfinity;
            int bestAction = oldAction;
            for (int a = 0; a < nA; a++)
            {
                double q = 0;
                foreach (var t in env.P[s, a])
                    q += t.Prob * (t.Reward + gamma * V[t.NextState] * (t.Done ? 0 : 1));
                if (q > best) { best = q; bestAction = a; }
            }
            policy[s] = bestAction;
            if (bestAction != oldAction) changed++;
        }

        history.Add((sweeps, changed));
        // Critere d'arret STRUCTUREL : plus aucune action ne change.
        if (changed == 0) return (V, policy, history);
    }
    return (V, policy, history);
}

In [10]:
void ReportPolicyIteration(string label, GridEnvironment e, double[] valuePolicy, int[] policyPolicy,
                           List<(int Sweeps, int Changed)> history, double[] valueStar, int[] policyStar)
{
    int totalSweeps = history.Sum(h => h.Sweeps);
    Console.WriteLine($"{label} — {history.Count} ameliorations, {totalSweeps} balayages d'evaluation au total.");
    Console.WriteLine($"  {"cycle",-8}{"balayages",-13}{"etats changes",-15}");
    for (int i = 0; i < history.Count; i++)
        Console.WriteLine($"  {(i + 1),-8}{history[i].Sweeps,-13}{history[i].Changed,-15}");

    double maxDiff = Enumerable.Range(0, e.NStates).Max(s => Math.Abs(valuePolicy[s] - valueStar[s]));
    bool identical = Enumerable.Range(0, e.NStates).All(s => policyPolicy[s] == policyStar[s]);
    Console.WriteLine($"  ecart maximal |V_pi - V*| : {maxDiff:0.00e+00}");
    Console.WriteLine($"  politique identique a celle de l'iteration de valeur : {(identical ? "oui" : "non")}");
    Console.WriteLine();
}

var (vPi, piPi, historyPi) = PolicyIteration(frozen, gamma: Gamma);
ReportPolicyIteration(frozen.Name, frozen, vPi, piPi, historyPi, vStar, piStar);

var (vPiSlip, piPiSlip, historyPiSlip) = PolicyIteration(frozenSlippery, gamma: Gamma);
ReportPolicyIteration(frozenSlippery.Name, frozenSlippery, vPiSlip, piPiSlip, historyPiSlip, vStarSlip, piStarSlip);

FrozenLake-v1 (is_slippery=False) — 7 ameliorations, 13 balayages d'evaluation au total.


  cycle   balayages    etats changes  


  1       1            3              


  2       2            2              


  3       2            3              


  4       2            3              


... (12 outputs left out)

  6       208          1              


  7       251          0              


  ecart maximal |V_pi - V*| : 3.27e-09


  politique identique a celle de l'iteration de valeur : oui


### Lecture du résultat

Les deux algorithmes trouvent la **même politique** sur les deux modèles — c'est la garantie théorique, et elle est vérifiée ici par comparaison directe, pas supposée (la ligne « politique identique » du rapport).

| Modèle | Itération de valeur | Itération de politique | Accord |
|---|---|---|---|
| non glissant | 7 balayages | 7 améliorations, **13** balayages d'évaluation | politique identique, écart $V$ de `0.00e+00` |
| glissant | **324** balayages | 7 améliorations, **976** balayages d'évaluation | politique identique, écart $V$ de `3.27e-09` |

Trois choses se lisent dans ces nombres.

**Le nombre d'améliorations est petit et plafonne.** Sept dans les deux cas, et la dernière ne change **aucune** action : c'est le critère d'arrêt structurel, pas un seuil. La colonne « états changés » montre la décroissance attendue — 2, 7, 6, 6, 2, 1, 0 sur le modèle glissant — et c'est cette décroissance, pas une tolérance numérique, qui termine l'algorithme.

**Le coût s'est déplacé, il n'a pas disparu.** Sur ces 16 états, l'itération de politique est **plus chère** en balayages que l'itération de valeur (976 contre 324) : chaque cycle paie l'évaluation complète d'une politique, et les premiers cycles évaluent des politiques très mauvaises. L'avantage classique de l'itération de politique — peu de cycles, donc peu de $\arg\max$ — n'apparaît que lorsque le nombre d'états grandit, où le nombre d'améliorations reste petit alors que les balayages de l'itération de valeur croissent avec l'horizon effectif. **Ce carnet ne mesure pas ce régime** : il ne faut pas lire ce tableau comme un classement général.

**L'écart résiduel de `3.27e-09` n'est pas un bug.** L'évaluation s'arrête quand `delta < theta = 1e-8`, donc $V^\pi$ est approchée, pas exacte. La politique, elle, est exacte : elle ne dépend que de l'ordre des valeurs, et un écart de $10^{-9}$ ne change aucun $\arg\max$.

Un dernier point, qui prépare la suite : les deux algorithmes ci-dessus **lisent la table de transitions $P$**. Ils sont *model-based*. La section suivante retire cette hypothèse, et c'est là que la difficulté change de nature.


## 5. Q-Learning — apprendre sans le modèle

Les sections 3 et 4 supposaient connaître $P$ et $R$. C'est une hypothèse forte : dans un vrai problème, la dynamique est inconnue et on ne dispose que d'**expériences** — des quintuplets $(s, a, r, s', \text{termine})$ observés.

Q-Learning apprend la fonction d'action-valeur $Q(s,a)$ par **différence temporelle**, sans jamais consulter le modèle :

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ \underbrace{r + \gamma \max_{a'} Q(s',a')}_{\text{cible TD}} - Q(s,a) \right]$$

Trois propriétés le distinguent d'une simple régression :

- **Il est hors-politique** (*off-policy*). La cible utilise $\max_{a'}$, pas l'action effectivement jouée : la valeur apprise est celle de la politique **optimale**, même si l'agent explore avec une politique différente. C'est le point que SARSA (section 6) renversera.
- **Il est incrémental.** Chaque transition corrige une seule case de la table, d'une fraction $\alpha$ de l'erreur. Aucune matrice à inverser, aucun système à résoudre.
- **Il converge** vers $Q^*$ si chaque couple $(s,a)$ est visité infiniment souvent et si $\alpha$ décroît aux conditions usuelles (Robbins-Monro). Avec un $\alpha$ constant — le choix fait ici — il converge vers un voisinage de $Q^*$ dont le rayon dépend de $\alpha$.

L'exploration est assurée par une politique **$\varepsilon$-gloutonne** : une action aléatoire avec probabilité $\varepsilon$, l'action de valeur maximale sinon. Sans elle, l'agent exploiterait indéfiniment la première action qui rapporte et n'apprendrait jamais les autres — la table resterait à zéro là où il n'est jamais allé.

L'environnement reste `FrozenLake` glissant : c'est le cas intéressant, puisqu'un pas peut glisser et que la récompense n'arrive qu'au but.


In [11]:
// Argmax avec egalites departagees au hasard : necessaire pour ne pas figer
// arbitrairement l'action 0 quand toute la ligne vaut zero au depart.
int ArgMax(double[] values, Random rng)
{
    double best = double.NegativeInfinity;
    var ties = new List<int>();
    for (int i = 0; i < values.Length; i++)
    {
        if (values[i] > best + 1e-12) { best = values[i]; ties.Clear(); ties.Add(i); }
        else if (Math.Abs(values[i] - best) <= 1e-12) ties.Add(i);
    }
    return ties.Count == 1 ? ties[0] : ties[rng.Next(ties.Count)];
}

int EpsilonGreedy(double[,] Q, int state, double epsilon, Random rng)
{
    int nA = Q.GetLength(1);
    if (rng.NextDouble() < epsilon) return rng.Next(nA);
    var row = new double[nA];
    for (int a = 0; a < nA; a++) row[a] = Q[state, a];
    return ArgMax(row, rng);
}

int GreedyAction(double[,] Q, int state)
{
    int nA = Q.GetLength(1);
    int best = 0;
    for (int a = 1; a < nA; a++)
        if (Q[state, a] > Q[state, best]) best = a;
    return best;
}

// max_a Q(s,a) : la graine du maximum est Q[s,0], JAMAIS zero. Initialiser a zero
// plafonne le maximum, ce qui est silencieux sur des valeurs positives et faux
// des que les valeurs sont negatives (CliffWalking, section 7).
double MaxAction(double[,] Q, int state)
{
    int nA = Q.GetLength(1);
    double best = Q[state, 0];
    for (int a = 1; a < nA; a++)
        if (Q[state, a] > best) best = Q[state, a];
    return best;
}

// Les visites sont rendues : sans elles, un echec d'apprentissage se constate
// mais ne s'explique pas. C'est le compteur qui montre la couverture reelle.
(double[,] Q, double[] Returns, int[] Steps, int[,] Visits) QLearning(
    GridEnvironment env, int episodes, double alpha, double gamma, double epsilon,
    int seed, int maxStepsPerEpisode = 200)
{
    var rng = new Random(seed);
    var Q = new double[env.NStates, env.NActions];
    var returns = new double[episodes];
    var steps = new int[episodes];
    var visits = new int[env.NStates, env.NActions];

    for (int episode = 0; episode < episodes; episode++)
    {
        int state = env.StartState;
        double total = 0;
        int t = 0;

        while (t < maxStepsPerEpisode)
        {
            int action = EpsilonGreedy(Q, state, epsilon, rng);
            visits[state, action]++;
            var (next, reward, done) = env.Step(state, action, rng);

            double bestNext = done ? 0 : MaxAction(Q, next);

            // La seule ligne qui apprend : la cible TD, puis la correction.
            Q[state, action] += alpha * (reward + gamma * bestNext - Q[state, action]);

            total += reward;
            state = next;
            t++;
            if (done) break;
        }

        returns[episode] = total;
        steps[episode] = t;
    }
    return (Q, returns, steps, visits);
}

### Anatomie de l'implémentation

Trois lignes portent toute la sémantique, et deux détails s'y cachent.

**`if (!done) bestNext = max_a Q[next, a]`** — quand l'épisode se termine, la cible est `reward` seul, sans terme d'avenir. L'oublier est l'erreur classique : la valeur d'un trou ou du but « fuirait » vers l'état d'où l'on vient, et $Q$ convergerait vers une valeur trop élevée, sans que rien ne signale l'erreur — l'algorithme *converge* toujours, simplement vers la mauvaise fonction.

**`t < maxStepsPerEpisode`** — sur la glace, l'agent qui joue au hasard erre longtemps avant d'atteindre le but : sans plafond de pas, un épisode peut durer des milliers de transitions et la boucle ne rend jamais la main. Le plafond est aussi ce qui distingue un épisode « réussi » d'un épisode « expiré » dans les statistiques.

Le reste est une politique $\varepsilon$-gloutonne et un `ArgMax` qui départage les égalités **au hasard**. Ce départage n'est pas cosmétique : au premier épisode, toute la ligne vaut zéro, et un `argmax` déterministe enverrait chaque état d'où l'on repart toujours vers la même action, ce qui biaise l'exploration initiale.


In [12]:
const int Episodes = 5000;
const double Epsilon = 0.1;

var (Q, returnsQl, stepsQl, _) = QLearning(frozenSlippery, Episodes, Alpha, Gamma, Epsilon, Seed);

double[] VFromQ(double[,] q, GridEnvironment e)
{
    var v = new double[e.NStates];
    for (int s = 0; s < e.NStates; s++)
        for (int a = 0; a < e.NActions; a++)
            if (q[s, a] > v[s]) v[s] = q[s, a];
    return v;
}

// La qualite d'une politique se mesure SANS exploration : epsilon = 0.
double EvaluateGreedy(GridEnvironment e, double[,] q, int episodes, int seed, int maxSteps = 200)
{
    var rng = new Random(seed);
    int wins = 0;
    for (int i = 0; i < episodes; i++)
    {
        int s = e.StartState;
        for (int t = 0; t < maxSteps; t++)
        {
            var (next, reward, done) = e.Step(s, GreedyAction(q, s), rng);
            s = next;
            if (done) { if (reward > 0) wins++; break; }
        }
    }
    return (double)wins / episodes;
}

// Les etats terminaux ne sont jamais quittes : Q y reste a zero et ne dit rien d'eux.
bool IsNonTerminal(GridEnvironment e, int s) => e.Cell(s) != 'H' && e.Cell(s) != 'G';

var vLearned = VFromQ(Q, frozenSlippery);
var policyLearned = Enumerable.Range(0, frozenSlippery.NStates).Select(s => GreedyAction(Q, s)).ToArray();

int window = 500;
double successTraining = returnsQl.Skip(Episodes - window).Average();
double successGreedy = EvaluateGreedy(frozenSlippery, Q, 2000, Seed + 1);
double meanSteps = stepsQl.Skip(Episodes - window).Average();

Console.WriteLine($"{Episodes} episodes, alpha = {Alpha}, gamma = {Gamma}, epsilon = {Epsilon}, graine {Seed}.");
Console.WriteLine($"Reussite en APPRENTISSAGE, {window} premiers episodes : {returnsQl.Take(window).Average():P1}");
Console.WriteLine($"Reussite en APPRENTISSAGE, {window} derniers     : {successTraining:P1}   (epsilon = {Epsilon}, donc 1 pas sur 10 est aleatoire)");
Console.WriteLine($"Reussite de la POLITIQUE gloutonne evaluee sans exploration (2000 episodes) : {successGreedy:P1}");
Console.WriteLine($"Pas moyens par episode d'apprentissage sur les {window} derniers : {meanSteps:0.0}");
Console.WriteLine();
Console.WriteLine("V(s) = max_a Q(s,a) appris, a comparer a V* de la section 3 :");
Console.WriteLine(RenderDoubleGrid(frozenSlippery, vLearned));
Console.WriteLine($"V(appris, depart) = {vLearned[frozenSlippery.StartState]:0.0000}   |   V*(depart) = {vStarSlip[frozenSlippery.StartState]:0.0000}");

var states = Enumerable.Range(0, frozenSlippery.NStates).Where(s => IsNonTerminal(frozenSlippery, s)).ToArray();
double maxGap = states.Max(s => Math.Abs(vLearned[s] - vStarSlip[s]));
Console.WriteLine($"Ecart maximal |V_appris - V*| sur les {states.Length} etats NON terminaux : {maxGap:0.0000}");
bool sameAsOptimal = states.All(s => policyLearned[s] == piStarSlip[s]);
Console.WriteLine($"Politique gloutonne identique a la politique optimale sur ces {states.Length} etats : {(sameAsOptimal ? "oui" : "non")}");

5000 episodes, alpha = 0.1, gamma = 0.99, epsilon = 0.1, graine 42.


Reussite en APPRENTISSAGE, 500 premiers episodes : 4.2 %


Reussite en APPRENTISSAGE, 500 derniers     : 39.4 %   (epsilon = 0.1, donc 1 pas sur 10 est aleatoire)


Reussite de la POLITIQUE gloutonne evaluee sans exploration (2000 episodes) : 81.8 %


Pas moyens par episode d'apprentissage sur les 500 derniers : 31.8


V(s) = max_a Q(s,a) appris, a comparer a V* de la section 3 :


      0.536   0.500   0.471   0.461
      0.555   0.000   0.356   0.000
      0.606   0.666   0.629   0.000
      0.000   0.754   0.871   0.000


V(appris, depart) = 0.5362   |   V*(depart) = 0.5420


Ecart maximal |V_appris - V*| sur les 11 etats NON terminaux : 0.0231


Politique gloutonne identique a la politique optimale sur ces 11 etats : oui


### Lecture du résultat

Deux chiffres de réussite cohabitent, et les confondre est l'erreur de lecture la plus courante : **39,4 %** pendant l'**apprentissage** et **81,8 %** pour la **politique finale**.

Les 39,4 % sont mesurés pendant l'entraînement, donc **avec exploration** : un pas sur dix est tiré au hasard (`epsilon = 0.1`). Sur une glace glissante, un pas aléatoire tous les dix pas suffit à faire tomber l'agent : il traverse la grille en une trentaine de pas, et une seule chute sur ce trajet annule l'épisode. Le taux plafonne donc bien en dessous de la qualité réelle de la politique apprise. Les 81,8 % mesurent cette même politique **sans exploration** (2000 épisodes, `epsilon = 0`) : c'est le seul chiffre qui parle de ce que l'agent **sait**. Un rapport qui n'annoncerait que 39,4 % sous-vendrait l'apprentissage d'un facteur deux.

**L'écart à l'optimum, et le piège des états terminaux.** `V(appris, depart) = 0.5362` contre `V*(depart) = 0.5420` : à trois millièmes de la valeur optimale. Mais la première version de cette cellule imprimait un écart maximal de **0,9514** et une politique « non identique ». Les deux mesures étaient fausses, et pour une raison qui mérite d'être écrite.

L'itération de valeur calcule `V*(s)` pour **les 16 états**, y compris les trous et le but : depuis un trou, la dynamique permet encore de bouger (récompense 0), donc `V*` y vaut ce qu'on gagnerait à en sortir — `V*(15) = 0.951` pour le but, `V*(5) = 0.555` pour le trou 1-1. Q-Learning, lui, **n'agit jamais** depuis un état terminal : l'épisode s'arrête à l'instant où l'on y arrive, donc `Q[s, ·]` reste **exactement zéro** pour les cinq états terminaux (5, 7, 11, 12, 15). Comparer les deux grilles sur 16 états revient à comparer une table qui répond à la question posée à une table qui répond à une **autre question** — et l'écart de 0,95 est le produit de cette confusion, pas un défaut d'apprentissage.

Restreint aux **11 états non terminaux**, où les deux formalismes parlent de la même chose : écart maximal **0,0231**, et la politique gloutonne **coïncide exactement** avec la politique optimale de la section 3. C'est le résultat attendu — sur 16 états, 5000 épisodes suffisent largement à identifier la politique optimale.

**Un seul modèle, une seule graine.** Ce chiffre vient d'un unique entraînement (`seed = 42`). Un run ne fait pas une mesure : la section suivante rejoue l'apprentissage sur quatre graines et balaie `epsilon`, `alpha` et `gamma` pour distinguer ce qui tient de ce qui doit au hasard de la graine.


## 6. Ce qui doit au hasard : quatre graines, trois hyperparamètres

Un entraînement réussi ne prouve rien. Q-Learning est **stochastique** : l'exploration tire au hasard, l'environnement glisse au hasard, et deux graines différentes produisent deux tables `Q` différentes. Un chiffre unique — « 81,8 % » — ne distingue pas une propriété de l'algorithme d'un coup de chance de la graine 42.

La règle de mesure est donc : **toute affirmation sur une performance se rejoue sur plusieurs graines**, et l'on rapporte la **moyenne avec son étendue** (`min`–`max`), jamais un point seul. Les quatre graines `0, 1, 7, 42` reviennent dans tous les balayages de cette section.

Trois hyperparamètres gouvernent l'apprentissage, et ils ne jouent pas le même rôle :

| Paramètre | Rôle | Ce qu'on attend |
|---|---|---|
| `epsilon` | part d'exploration | trop bas : se fige sur une politique médiocre ; trop haut : n'exploite jamais ce qu'il sait |
| `alpha` | pas d'apprentissage | trop bas : n'oublie jamais l'erreur initiale ; trop haut : la table oscille et n'oublie pas le bruit |
| `gamma` | horizon | bas : préfère un gain proche ; haut : valorise l'atteinte du but, fût-elle lointaine — mais aussi plus de variance |

L'ordre des balayages n'est pas indifférent. On commence par `epsilon`, les deux autres fixés aux valeurs partagées (`alpha = 0.1`, `gamma = 0.99`) : c'est le paramètre dont on soupçonne déjà qu'il explique l'échec de la graine 7. Les deux balayages suivants fixent ensuite **`epsilon = 0.2`**, la valeur que le premier aura montrée comme la seule sans échec — étudier `alpha` sur une base qui échoue une graine sur quatre confondrait deux causes.


In [13]:
int[] SweepSeeds = { 0, 1, 7, 42 };
const int SweepEpisodes = 5000;
const int EvalEpisodes = 1000;

double[] MovingAverage(double[] values, int window)
{
    var result = new double[values.Length / window];
    for (int i = 0; i < result.Length; i++)
        result[i] = values.Skip(i * window).Take(window).Average();
    return result;
}

var curves = new List<(string Label, double[] Xs, double[] Ys, string Color)>();
string[] curveColors = { "#1f77b4", "#d62728", "#2ca02c", "#9467bd" };
double[] episodeGrid = null;

for (int i = 0; i < SweepSeeds.Length; i++)
{
    var (q, rets, _, _) = QLearning(frozenSlippery, SweepEpisodes, Alpha, Gamma, Epsilon, SweepSeeds[i]);
    double[] smoothed = MovingAverage(rets, 100);
    episodeGrid ??= Enumerable.Range(0, smoothed.Length).Select(k => (double)((k + 1) * 100)).ToArray();
    curves.Add(($"graine {SweepSeeds[i]}", episodeGrid, smoothed, curveColors[i]));
    double greedy = EvaluateGreedy(frozenSlippery, q, EvalEpisodes, SweepSeeds[i] + 7919);
    Console.WriteLine($"graine {SweepSeeds[i],2} : glouton final {greedy,6:P1}   |   derniere fenetre d'apprentissage {smoothed[^1]:P1}");
}

PlotLines("Reussite glissante pendant l'apprentissage (fenetre de 100 episodes)",
          "episode", "taux de reussite", curves).Display();

graine  0 : glouton final 81.7 %   |   derniere fenetre d'apprentissage 41.0 %


graine  1 : glouton final 78.7 %   |   derniere fenetre d'apprentissage 50.0 %


graine  7 : glouton final 29.2 %   |   derniere fenetre d'apprentissage 33.0 %


graine 42 : glouton final 81.0 %   |   derniere fenetre d'apprentissage 41.0 %


Reussite glissante pendant l'apprentissage (fenetre de 100 episodes) -0.0408 0.1071 0.255 0.4029 0.5508 100 1080 2060 3040 4020 5000 episode taux de reussite <polyline points="74,300.3 83.1,286.3 92.3,291 101.4,300.3 110.6,277 119.7,267.6 128.9,258.3 138,267.6 147.1,211.7 156.3,193 165.4,183.7 174.6,183.7 183.7,137 192.9,104.4 202,155.7 211.1,151 220.3,132.3 229.4,151 238.6,160.3 247.7,165 256.9,165 266,109 275.1,67 284.3,81 293.4,90.4 302.6,81 311.7,141.7 320.9,109 330,81 339.1,160.3 348.3,113.7 357.4,109 366.6,127.7 375.7,151 384.9,113.7 394,127.7 403.1,127.7 412.3,85.7 421.4,99.7 430.6,95 439.7,113.7 448.9,141.7 458,132.3 467.1,123 476.3,90.4 485.4,99.7 494.6,109 503.7,99.7 512.9,113.7 522,113.7" fill="none" stroke="#1f77b4" stroke-width="2.2" stroke-linejoin="round"/><polyline points="74,291 83.1,300.3 92.3,295.6 101.4,300.3 110.6,277 119.7,281.6 128.9,272.3 138,286.3 147.1,239.7 156.3,263 165.4,267.6 174.6,258.3 183.7,267.6 192.9,267.6 202,244.3 211.1,263 220.3,221 229.4,253.6 238.6,263 247.7,221 256.9,211.7 266,141.7 275.1,113.7 284.3,113.7 293.4,113.7 302.6,151 311.7,113.7 320.9,141.7 330,141.7 339.1,90.4 348.3,90.4 357.4,104.4 366.6,95 375.7,123 384.9,95 394,99.7 403.1,183.7 412.3,109 421.4,109 430.6,81 439.7,123 448.9,146.3 458,160.3 467.1,169.7 476.3,90.4 485.4,95 494.6,137 503.7,109 512.9,132.3 522,71.7" fill="none" stroke="#d62728" stroke-width="2.2" stroke-linejoin="round"/><polyline points="74,305 83.1,305 92.3,305 101.4,286.3 110.6,300.3 119.7,300.3 128.9,286.3 138,300.3 147.1,291 156.3,249 165.4,169.7 174.6,188.3 183.7,216.3 192.9,146.3 202,151 211.1,169.7 220.3,151 229.4,113.7 238.6,118.4 247.7,141.7 256.9,127.7 266,127.7 275.1,132.3 284.3,146.3 293.4,76.4 302.6,146.3 311.7,127.7 320.9,104.4 330,99.7 339.1,127.7 348.3,127.7 357.4,109 366.6,99.7 375.7,127.7 384.9,132.3 394,109 403.1,85.7 412.3,146.3 421.4,104.4 430.6,123 439.7,137 448.9,127.7 458,113.7 467.1,99.7 476.3,141.7 485.4,113.7 494.6,118.4 503.7,137 512.9,104.4 522,151" fill="none" stroke="#2ca02c" stroke-width="2.2" stroke-linejoin="round"/><polyline points="74,295.6 83.1,295.6 92.3,281.6 101.4,272.3 110.6,281.6 119.7,291 128.9,286.3 138,207 147.1,113.7 156.3,127.7 165.4,137 174.6,127.7 183.7,160.3 192.9,99.7 202,141.7 211.1,104.4 220.3,127.7 229.4,104.4 238.6,123 247.7,85.7 256.9,160.3 266,127.7 275.1,85.7 284.3,127.7 293.4,132.3 302.6,104.4 311.7,109 320.9,109 330,132.3 339.1,71.7 348.3,132.3 357.4,99.7 366.6,95 375.7,95 384.9,132.3 394,113.7 403.1,160.3 412.3,146.3 421.4,132.3 430.6,132.3 439.7,90.4 448.9,123 458,81 467.1,99.7 476.3,76.4 485.4,132.3 494.6,104.4 503.7,132.3 512.9,123 522,113.7" fill="none" stroke="#9467bd" stroke-width="2.2" stroke-linejoin="round"/> graine 0 graine 1 graine 7 graine 42

### Lecture du résultat

Les quatre courbes montent, puis **plafonnent autour de 40 à 50 %** — et l'on sait déjà pourquoi, depuis la section 5 : ce sont les taux mesurés **avec** exploration. Ce plafond n'est pas une limite de l'algorithme mais le prix de `epsilon = 0.1`, payé à chaque pas.

La dispersion entre graines est en revanche réelle et instructive. Trois graines terminent entre 78,7 % et 81,7 % de réussite gloutonne ; la **graine 7 reste à 29,2 %**. Ce n'est pas du bruit de mesure : c'est une **politique différente**, et une politique fausse.

La grille de la graine 7 le montre sans ambiguïté. Depuis `S` (état 0, coin supérieur gauche), sa politique gloutonne est **gauche** — or la case à gauche de `S` est hors de la grille, et une action hors grille laisse l'agent sur place. La politique apprise consiste donc à **pousser dans le mur** : elle n'avance que par les glissements aléatoires de la glace, qui la font dériver 1 fois sur 3 vers le bas. `V(0)` y vaut 0,495 contre 0,542 à l'optimum, et l'écart maximal aux états non terminaux atteint 0,31 (contre 0,023 pour la graine 42).

C'est un **optimum local**, et son mécanisme est celui de l'exploration/exploitation. Au tout début, la table `Q` est nulle partout. Le départage aléatoire des égalités envoie un jour l'agent à gauche ; comme cette action laisse l'agent sur `S`, elle revient sans cesse et son estimation `Q(S, gauche)` monte — non parce qu'elle est bonne, mais parce qu'elle est **fréquemment réessayée**. Avec `epsilon = 0.1`, neuf pas sur dix exploitent cette préférence naissante ; l'échantillon de l'action « droite », lui, reste rare, et son estimation reste basse. La politique se fige sur la mauvaise action.

Ce cas est le meilleur argument contre le chiffre unique : annoncer « Q-Learning réussit à 81 % sur FrozenLake » à partir de la seule graine 42 aurait caché qu'**une graine sur quatre échoue**. Le balayage qui suit mesure si plus d'exploration répare ce cas, ou s'il faut autre chose.


In [14]:
double[] epsilons = { 0.01, 0.05, 0.1, 0.2, 0.3, 0.5 };

(double Greedy, int[,] Visits) GreedyAfterTraining(GridEnvironment env, double alpha, double gamma, double epsilon, int seed)
{
    var (q, _, _, visits) = QLearning(env, SweepEpisodes, alpha, gamma, epsilon, seed);
    return (EvaluateGreedy(env, q, EvalEpisodes, seed + 7919), visits);
}

var epsilonRows = new List<(double Eps, double Mean, double Min, double Max, double[] Per)>();
foreach (var eps in epsilons)
{
    double[] per = SweepSeeds.Select(s => GreedyAfterTraining(frozenSlippery, Alpha, Gamma, eps, s).Greedy).ToArray();
    epsilonRows.Add((eps, per.Average(), per.Min(), per.Max(), per));
    Console.WriteLine($"epsilon = {eps,-5:0.00}  moyenne {per.Average():P1}  [{per.Min():P1} - {per.Max():P1}]   par graine : {string.Join("   ", per.Select(v => v.ToString("P1")))}");
}

var epsilonSeries = new List<(string Label, double[] Xs, double[] Ys, string Color)>
{
    ("moyenne", epsilons, epsilonRows.Select(r => r.Mean).ToArray(), "#1f77b4"),
    ("minimum", epsilons, epsilonRows.Select(r => r.Min).ToArray(), "#d62728"),
    ("maximum", epsilons, epsilonRows.Select(r => r.Max).ToArray(), "#2ca02c"),
};
PlotLines("Reussite gloutonne finale selon epsilon (4 graines)",
          "epsilon", "taux de reussite", epsilonSeries).Display();

epsilon = 0.01   moyenne 36.2 %  [15.6 % - 58.2 %]   par graine : 35.1 %   35.8 %   58.2 %   15.6 %


epsilon = 0.05   moyenne 77.4 %  [75.1 % - 80.5 %]   par graine : 75.1 %   77.6 %   80.5 %   76.5 %


epsilon = 0.10   moyenne 67.7 %  [29.2 % - 81.7 %]   par graine : 81.7 %   78.7 %   29.2 %   81.0 %


epsilon = 0.20   moyenne 81.3 %  [80.5 % - 81.7 %]   par graine : 81.7 %   81.3 %   80.5 %   81.6 %


epsilon = 0.30   moyenne 74.2 %  [53.0 % - 81.7 %]   par graine : 81.7 %   81.0 %   53.0 %   81.0 %


epsilon = 0.50   moyenne 80.0 %  [77.2 % - 81.3 %]   par graine : 77.2 %   81.3 %   80.5 %   81.0 %


Reussite gloutonne finale selon epsilon (4 graines) 0.1031 0.2948 0.4865 0.6782 0.8699 0.01 0.108 0.206 0.304 0.402 0.5 epsilon taux de reussite moyenne minimum maximum

**Pourquoi un $\varepsilon$ trop bas échoue — mesuré au lieu d'être supposé.** Le balayage précédent donne un taux de réussite par valeur de $\varepsilon$, mais pas la cause. La cellule suivante compte, état par état, combien de fois l'action optimale a été **essayée** : un état dont l'action optimale n'est jamais jouée ne peut pas être appris, quelle que soit la valeur de $\alpha$. C'est la dispersion de ces comptes qui est le résultat — pas le taux moyen.

In [15]:
// Pourquoi epsilon trop bas echoue : compter les visites, pas les supposer.
string[] actionLetters = { "G", "B", "D", "H" };
var (qLow, _, _, lowVisits) = QLearning(frozenSlippery, SweepEpisodes, Alpha, Gamma, 0.05, 0);
var (qHigh, _, _, highVisits) = QLearning(frozenSlippery, SweepEpisodes, Alpha, Gamma, 0.20, 0);
var polLow = Enumerable.Range(0, frozenSlippery.NStates).Select(s => GreedyAction(qLow, s)).ToArray();
var polHigh = Enumerable.Range(0, frozenSlippery.NStates).Select(s => GreedyAction(qHigh, s)).ToArray();

Console.WriteLine("Visites de l'action OPTIMALE de chaque etat — graine 0, 5000 episodes");
Console.WriteLine("etat | opt | visites eps=0.05 | visites eps=0.20 | choix eps=0.05 | choix eps=0.20");
Console.WriteLine("-----+-----+------------------+------------------+----------------+---------------");
foreach (var s in Enumerable.Range(0, frozenSlippery.NStates).Where(x => IsNonTerminal(frozenSlippery, x)))
{
    int opt = piStarSlip[s];
    string low = polLow[s] == opt ? "ok" : $"{actionLetters[polLow[s]]} FAUX";
    string high = polHigh[s] == opt ? "ok" : $"{actionLetters[polHigh[s]]} FAUX";
    Console.WriteLine($"{s,4} |  {actionLetters[opt]}  | {lowVisits[s, opt],16} | {highVisits[s, opt],16} | {low,14} | {high,13}");
}

int maxLow = Enumerable.Range(0, frozenSlippery.NStates).Where(x => IsNonTerminal(frozenSlippery, x)).Max(x => lowVisits[x, piStarSlip[x]]);
int minLow = Enumerable.Range(0, frozenSlippery.NStates).Where(x => IsNonTerminal(frozenSlippery, x)).Min(x => lowVisits[x, piStarSlip[x]]);
Console.WriteLine();
Console.WriteLine($"eps = 0.05 : action optimale essayee de {minLow} a {maxLow} fois selon l'etat (facteur {maxLow / Math.Max(1, minLow)})");
Console.WriteLine($"eps = 0.05 : {Enumerable.Range(0, frozenSlippery.NStates).Count(s => IsNonTerminal(frozenSlippery, s) && polLow[s] != piStarSlip[s])} etats non optimaux sur 11   |   glouton {EvaluateGreedy(frozenSlippery, qLow, 2000, 1):P1}");
Console.WriteLine($"eps = 0.20 : {Enumerable.Range(0, frozenSlippery.NStates).Count(s => IsNonTerminal(frozenSlippery, s) && polHigh[s] != piStarSlip[s])} etats non optimaux sur 11   |   glouton {EvaluateGreedy(frozenSlippery, qHigh, 2000, 1):P1}");

Visites de l'action OPTIMALE de chaque etat — graine 0, 5000 episodes


etat | opt | visites eps=0.05 | visites eps=0.20 | choix eps=0.05 | choix eps=0.20


-----+-----+------------------+------------------+----------------+---------------


   0 |  G  |            39758 |            27796 |             ok |            ok


   1 |  H  |              359 |             4941 |             ok |            ok


   2 |  H  |               36 |             3167 |         B FAUX |            ok


... (7 outputs left out)

  14 |  B  |             5551 |             2662 |             ok |            ok


eps = 0.05 : action optimale essayee de 11 a 39758 fois selon l'etat (facteur 3614)


eps = 0.05 : 2 etats non optimaux sur 11   |   glouton 74.7 %


eps = 0.20 : 1 etats non optimaux sur 11   |   glouton 81.4 %


### Lecture du résultat

La courbe n'est **pas** une fonction croissante d'`epsilon`, et c'est précisément ce qu'il faut lire.

| `epsilon` | moyenne | étendue | lecture |
|---|---|---|---|
| 0,01 | 36,2 % | 15,6 – 58,2 % | trop peu d'exploration : deux graines échouent, une troisième plafonne |
| 0,05 | 77,4 % | 75,1 – 80,5 % | stable, mais systématiquement sous l'optimum |
| 0,10 | 67,7 % | **29,2** – 81,7 % | la moyenne cache l'échec de la graine 7 |
| 0,20 | **81,3 %** | 80,5 – 81,7 % | seule valeur sans échec à 5000 épisodes |
| 0,30 | 74,2 % | **53,0** – 81,7 % | une graine partiellement bloquée |
| 0,50 | 80,0 % | 77,2 – 81,3 % | beaucoup d'exploration, convergence seulement plus lente |

**Le mécanisme est mesuré, pas supposé.** Le tableau de couverture compte, pour chaque état, combien de fois l'action **optimale** a été essayée. À `epsilon = 0.05`, cette action est tentée **39 758 fois** depuis l'état 0 et **11 fois** depuis l'état 3 — un facteur 3 614. Aux deux états les moins visités (2 et 3), la politique gloutonne choisit une action fausse (`B` et `G` au lieu de `H`), ce qui coûte environ 6 points de réussite. À `epsilon = 0.20`, ces deux états passent de 36 et 11 visites à 3 167 et 3 400 : la politique y devient correcte.

Le détail vaut d'être souligné : à `epsilon = 0.20` la politique conserve **une** action fausse, à l'état 6 — et cela ne coûte presque rien, parce que l'état 6 n'est **pas sur le chemin optimal**. Une politique n'est pas jugée au nombre d'états où elle diffère de l'optimum, mais au **coût** de ces différences. C'est aussi la raison pour laquelle `epsilon = 0.05` (2 états faux, tous deux sur le chemin) réussit moins que `epsilon = 0.20` (1 état faux, hors chemin).

**Pourquoi un `epsilon` élevé n'empoisonne-t-il pas les valeurs ?** La cible de mise à jour est `r + gamma · max_a Q(s', a)`, et ce maximum **ne dépend pas** de l'action effectivement jouée ensuite. C'est la propriété **hors-politique** de Q-Learning : la politique d'exploration ne biaise donc pas les estimations, elle ne fait que décider **quelles paires `(s, a)` sont échantillonnées**. D'où le résultat contre-intuitif mais mesuré : `epsilon = 0.5` atteint 80 %, presque autant que 0,2, alors qu'un agent qui jouerait au hasard la moitié du temps semblerait « mauvais ». Il explore simplement mieux, et converge plus lentement.

**Ce qu'il ne faut pas conclure.** Avec quatre graines, une moyenne à ±2 points n'est pas une mesure fine : les valeurs 77,4 / 81,3 / 80,0 de `epsilon` = 0,05 / 0,2 / 0,5 ne se départagent pas. La seule affirmation robuste est **qualitative** : sous 5 000 épisodes, un `epsilon` trop bas (≤ 0,01) échoue par manque de couverture, et 0,2 est la seule valeur qui ne produise **aucun** échec sur les quatre graines. Écrire « le meilleur `epsilon` est 0,20 » serait surinterpréter une expérience à quatre points.


In [16]:
double[] alphas = { 0.01, 0.05, 0.1, 0.3, 0.5, 0.9 };
const double ExplorationFixed = 0.2;

var alphaRows = new List<(double A, double Mean, double Min, double Max)>();
foreach (var a in alphas)
{
    double[] per = SweepSeeds.Select(s => GreedyAfterTraining(frozenSlippery, a, Gamma, ExplorationFixed, s).Greedy).ToArray();
    alphaRows.Add((a, per.Average(), per.Min(), per.Max()));
    Console.WriteLine($"alpha = {a,-5:0.00}  moyenne {per.Average():P1}  [{per.Min():P1} - {per.Max():P1}]   par graine : {string.Join("   ", per.Select(v => v.ToString("P1")))}");
}

var alphaSeries = new List<(string Label, double[] Xs, double[] Ys, string Color)>
{
    ("moyenne", alphas, alphaRows.Select(r => r.Mean).ToArray(), "#1f77b4"),
    ("minimum", alphas, alphaRows.Select(r => r.Min).ToArray(), "#d62728"),
    ("maximum", alphas, alphaRows.Select(r => r.Max).ToArray(), "#2ca02c"),
};
PlotLines("Reussite gloutonne finale selon alpha (4 graines, epsilon = 0.2)",
          "alpha", "taux de reussite", alphaSeries).Display();

alpha = 0.01   moyenne 15.7 %  [2.2 % - 51.4 %]   par graine : 6.2 %   2.2 %   2.8 %   51.4 %


alpha = 0.05   moyenne 80.4 %  [77.2 % - 81.7 %]   par graine : 81.7 %   81.3 %   77.2 %   81.6 %


alpha = 0.10   moyenne 81.3 %  [80.5 % - 81.7 %]   par graine : 81.7 %   81.3 %   80.5 %   81.6 %


alpha = 0.30   moyenne 58.9 %  [16.5 % - 80.9 %]   par graine : 80.9 %   74.6 %   63.5 %   16.5 %


alpha = 0.50   moyenne 63.9 %  [24.9 % - 80.5 %]   par graine : 78.0 %   24.9 %   80.5 %   72.0 %


alpha = 0.90   moyenne 18.6 %  [0.0 % - 54.9 %]   par graine : 0.0 %   11.0 %   8.6 %   54.9 %


Reussite gloutonne finale selon alpha (4 graines, epsilon = 0.2) -0.0654 0.1716 0.4085 0.6454 0.8824 0.01 0.188 0.366 0.544 0.722 0.9 alpha taux de reussite moyenne minimum maximum

### Lecture du résultat

`alpha` dessine cette fois la courbe en cloche attendue, et **les deux extrémités échouent** :

| `alpha` | moyenne | étendue |
|---|---|---|
| 0,01 | 15,7 % | 2,2 – 51,4 % |
| 0,05 | 80,4 % | 77,2 – 81,7 % |
| 0,10 | **81,3 %** | 80,5 – 81,7 % |
| 0,30 | 58,9 % | 16,5 – 80,9 % |
| 0,50 | 63,9 % | 24,9 – 80,5 % |
| 0,90 | 18,6 % | **0,0** – 54,9 % |

**Trop petit, l'erreur initiale ne s'efface pas.** `alpha` est le taux d'oubli de l'estimation : `Q ← Q + alpha · (cible − Q)` fait du nouveau `Q` une moyenne pondérée entre l'ancienne valeur et la nouvelle observation, avec une fenêtre d'environ `1/alpha` échantillons. À `alpha = 0.01`, cette fenêtre fait une centaine d'échantillons, mais la valeur doit **remonter** du but vers le départ, six pas en amont, et chaque étape de cette remontée coûte un facteur `alpha`. Sur 5 000 épisodes, la propagation n'a pas le temps d'aboutir : une graine finit à 2,2 %, c'est-à-dire **rien appris** — sa politique est encore celle du hasard.

**Trop grand, la table devient un échantillon.** À `alpha = 0.9`, chaque mise à jour remplace presque entièrement l'estimation par la dernière cible observée. Or cette cible est bruitée : la glace glisse, et l'exploration joue 20 % du temps au hasard. La table finale n'est plus une espérance, c'est un **instantané du bruit des derniers épisodes** — et une graine termine à 0 %, alors qu'elle avait appris quelque chose en cours de route. C'est un point de méthode qui vaut au-delà de ce carnet : **une politique se lit sur une table stabilisée, pas sur le dernier point d'une trajectoire**.

Entre les deux, le plateau 0,05–0,10 est large. Les moyennes de 0,3 et 0,5 (58,9 % et 63,9 %) ne s'ordonnent pas de façon monotone : avec quatre graines et une variance inter-graines de plus de 50 points, cet ordre n'est pas interprétable. Ce qui est mesuré, c'est la **forme** — les deux bouts échouent, le milieu tient — pas la position exacte du sommet.


In [17]:
double[] gammas = { 0.5, 0.9, 0.95, 0.99, 0.999 };
int[] budgets = { 500, 2000, 5000 };

double GreedyAt(GridEnvironment env, double alpha, double gamma, double epsilon, int seed, int episodes)
{
    var (q, _, _, _) = QLearning(env, episodes, alpha, gamma, epsilon, seed);
    return EvaluateGreedy(env, q, EvalEpisodes, seed + 7919);
}

Console.WriteLine("Reussite gloutonne moyenne (4 graines) selon gamma et le budget d'episodes");
Console.WriteLine("gamma |    500 ep |   2000 ep |   5000 ep |   V*(depart) par iteration de valeur");
Console.WriteLine("------+-----------+-----------+-----------+--------------------------------------");
foreach (var g in gammas)
{
    var cells = budgets.Select(b => SweepSeeds.Average(s => GreedyAt(frozenSlippery, Alpha, g, ExplorationFixed, s, b))).ToArray();
    var (viRaw, _, _) = ValueIteration(frozenSlippery, gamma: g);
    Console.WriteLine($"{g,-5:0.000} | {cells[0],8:P1}  | {cells[1],8:P1}  | {cells[2],8:P1}  | {viRaw[frozenSlippery.StartState]:0.0000}");
}

Reussite gloutonne moyenne (4 graines) selon gamma et le budget d'episodes


gamma |    500 ep |   2000 ep |   5000 ep |   V*(depart) par iteration de valeur


------+-----------+-----------+-----------+--------------------------------------


Convergence a l'iteration 18 (delta = 6.42e-09).


0.500 |    7.7 %  |   10.0 %  |   15.5 %  | 0.0004


Convergence a l'iteration 85 (delta = 9.02e-09).


... (3 outputs left out)

Convergence a l'iteration 324 (delta = 9.69e-09).


0.990 |   23.8 %  |   78.4 %  |   81.3 %  | 0.5420


Convergence a l'iteration 438 (delta = 9.99e-09).


0.999 |   26.4 %  |   80.1 %  |   60.6 %  | 0.7855


**Ce qui décide d'une politique n'est pas la valeur, mais l'écart entre actions.** Le balayage précédent mesure la réussite selon $\gamma$ ; il ne dit pas pourquoi un $\gamma$ plus grand cesse d'aider. La cellule suivante compare la table apprise à $V^*$ — écart moyen, écart maximal, et surtout l'écart entre les quatre actions d'un **même** état : quand cet écart tombe sous le bruit d'échantillonnage, la politique gloutonne devient un tirage au sort entre actions équivalentes.

In [18]:
// Ce qui decide d'une politique n'est pas la VALEUR, mais l'ECART entre actions
// compare au bruit d'echantillonnage. On mesure cet ecart, par gamma, graine 42.
double[] gammaGrid = { 0.5, 0.9, 0.95, 0.99, 0.999 };
var nonTerminal = Enumerable.Range(0, frozenSlippery.NStates).Where(s => IsNonTerminal(frozenSlippery, s)).ToArray();
double Spread(double[,] q, int s) => Enumerable.Range(0, 4).Max(a => q[s, a]) - Enumerable.Range(0, 4).Min(a => q[s, a]);

Console.WriteLine("gamma | ecart Q moyen (11 etats) | ecart Q max | ecart Q a l'etat 0 | reussite gloutonne");
Console.WriteLine("------+--------------------------+-------------+--------------------+--------------------");
var gammaTables = new Dictionary<double, double[,]>();
foreach (var g in gammaGrid)
{
    var (q, _, _, _) = QLearning(frozenSlippery, SweepEpisodes, Alpha, g, ExplorationFixed, 42);
    gammaTables[g] = q;
    double mean = nonTerminal.Average(s => Spread(q, s));
    double max = nonTerminal.Max(s => Spread(q, s));
    Console.WriteLine($"{g,-5:0.000} | {mean,24:0.00000} | {max,11:0.00000} | {Spread(q, 0),18:0.00000} | {EvaluateGreedy(frozenSlippery, q, 1000, 7),18:P1}");
}

// Etude de cas : gamma = 0.999, graine 42. Les quatre actions de l'etat 0 tiennent
// dans 5e-4 : le choix glouton se decide sur du bruit, pas sur une preference apprise.
var qExtreme = gammaTables[0.999];
string[] arrowNames = { "G", "B", "D", "H" };
Console.WriteLine();
Console.WriteLine("gamma = 0.999, graine 42 — Q(etat 0) : " + string.Join("   ", Enumerable.Range(0, 4).Select(a => $"{arrowNames[a]}={qExtreme[0, a]:0.000000}")));
Console.WriteLine($"action gloutonne a l'etat 0 : {arrowNames[GreedyAction(qExtreme, 0)]}");
Console.WriteLine("politique obtenue :");
for (int r = 0; r < frozenSlippery.Rows; r++)
{
    var line = new StringBuilder();
    for (int c = 0; c < frozenSlippery.Cols; c++)
    {
        int s = r * frozenSlippery.Cols + c;
        char cell = frozenSlippery.Cell(s);
        line.Append((cell == 'H' || cell == 'G' ? cell.ToString() : arrowNames[GreedyAction(qExtreme, s)]).PadLeft(4));
    }
    Console.WriteLine(line.ToString());
}
Console.WriteLine($"reussite de cette politique : {EvaluateGreedy(frozenSlippery, qExtreme, 1000, 7):P1}");

gamma | ecart Q moyen (11 etats) | ecart Q max | ecart Q a l'etat 0 | reussite gloutonne


------+--------------------------+-------------+--------------------+--------------------


0.500 |                  0.02960 |     0.14815 |            0.00012 |             17.2 %


0.900 |                  0.07430 |     0.33253 |            0.00456 |             48.7 %


0.950 |                  0.12057 |     0.29241 |            0.03377 |             79.3 %


0.990 |                  0.20320 |     0.33550 |            0.04715 |             82.0 %


... (5 outputs left out)

   G   H   G   H


   H   B   G   H


   H   D   B   G


reussite de cette politique : 0.0 %


### Lecture du résultat

| `gamma` | 500 ép. | 2 000 ép. | 5 000 ép. | `V*(départ)` | écart Q à l'état 0 |
|---|---|---|---|---|---|
| 0,50 | 7,7 % | 10,0 % | 15,5 % | 0,0004 | 0,0001 |
| 0,90 | 12,7 % | 55,9 % | 52,7 % | 0,0689 | 0,0046 |
| 0,95 | 14,2 % | 58,5 % | 70,8 % | 0,1805 | 0,0338 |
| 0,99 | 23,8 % | 78,4 % | **81,3 %** | 0,5420 | 0,0472 |
| 0,999 | 26,4 % | 80,1 % | **60,6 %** | 0,7855 | **0,0005** |

**`V*` n'est pas un taux de réussite.** À `gamma = 0.5`, `V*(départ) = 0.0004` — un chiffre qui ne ressemble à rien — alors que la politique optimale pour ce `gamma` atteint le but **45,3 %** du temps (mesuré sur 20 000 épisodes). `V*` est une **somme actualisée** : elle dépend de la récompense *et* de l'instant où elle arrive. Le même but, la même grille, et une valeur mille fois plus petite simplement parce que `0.5^5 = 0.03`. Confondre `V*` et une probabilité est l'erreur de lecture la plus fréquente sur ce type de tableau.

**Le discount fixe un compromis, mesuré.** Sous la politique optimale à `gamma = 0.99`, un épisode réussi dure **47 pas en moyenne** — la glace fait dériver l'agent bien au-delà des six pas du chemin le plus court, et le discount de 0,99 tolère cette errance. Sous la politique optimale à `gamma = 0.5`, le même calcul donne **29,5 pas** et 45,3 % de réussite : elle **arrive moins souvent, mais plus vite**. `gamma` n'est donc pas un « horizon plus ou moins long » dans l'abstrait : c'est le prix relatif du temps, et il change réellement la politique optimale, pas seulement l'échelle des valeurs.

**L'instrument qui explique tout : l'écart entre actions.** La colonne de droite mesure, à l'état de départ, l'écart entre la meilleure et la pire action de la table `Q` apprise. Il culmine à **0,0472** pour `gamma = 0.99`, exactement la valeur où la politique apprise coïncide avec l'optimum du modèle — et il s'effondre aux deux extrémités. Or c'est cet écart, et non la valeur absolue, qui décide du choix glouton. Trop petit, la politique est tirée au hasard du bruit d'échantillonnage.

**L'étude de cas `gamma = 0.999` le montre sans ambiguïté.** Les quatre actions de l'état 0 y valent `0.7769 / 0.7768 / 0.7772 / 0.7773` : elles tiennent dans **5 × 10⁻⁴**. Le maximum est `H` (aller vers le haut), qui depuis le coin supérieur gauche est un **mur** — l'agent reste sur place indéfiniment, et la politique réussit **0,0 %**. Rien, dans la table, ne distingue « monter dans le mur » de « descendre vers le but » : à `gamma` aussi proche de 1, la valeur de toutes les actions se vaut presque, parce que le but finit par être atteint quelle que soit la politique et que 0,999 par pas efface la différence entre six et cinquante pas.

Cela éclaire d'un jour nouveau le **80,1 % mesuré à 2 000 épisodes** pour ce même `gamma` : c'était déjà une politique de ce type — pousser vers un mur et laisser la glace dériver. L'écart entre actions y valait 0,011, la table était déjà dégénérée, et la dérive tombait par chance vers le but. **Un taux de réussite n'est pas une preuve d'apprentissage** : il faut regarder la table qui le produit. C'est la même leçon que la graine 7 de la section précédente, sous une autre forme.

Enfin, la non-monotonie du budget à `gamma = 0.999` (80,1 % à 2 000, 60,6 % à 5 000) vient d'**une seule graine** : `g0 = 80,9`, `g1 = 81,0`, `g7 = 80,5`, `g42 = 0,0 %`. La moyenne à quatre graines cache l'effondrement, exactement comme à `epsilon = 0.1`. Une moyenne sans son étendue n'est pas un résultat.

**Ce qui reste.** `gamma = 0.99` est la seule valeur qui produise une politique **identique à l'optimum du modèle** (section 5) et un écart entre actions qui domine le bruit. Ce n'est pas un réglage universel : c'est le résultat, sur cette grille, de la comparaison entre l'écart vrai entre actions et le bruit d'une table apprise en 5 000 épisodes.


## 7. CliffWalking : le risque, et la différence entre Q-Learning et SARSA

FrozenLake récompense l'arrivée et ne dit rien du **chemin**. CliffWalking pose l'autre question : à gauche le chemin le plus court longe une falaise, à droite un détour sûr. La falaise ne tue pas — elle rend `-100` et renvoie au départ — mais elle rend le raccourci tentant.

- grille `4 x 12`, départ en bas à gauche, but en bas à droite ;
- chaque pas coûte `-1` ; marcher dans la falaise coûte `-100` et termine l'épisode (convention Gymnasium) ;
- actions : `0 = haut`, `1 = droite`, `2 = bas`, `3 = gauche` — **attention**, ce n'est pas l'ordre de FrozenLake, où `0 = gauche` ;
- environnement **déterministe**, donc toute la stochasticité vient de l'exploration.

Le chemin optimal coûte `-13` (12 pas le long du bord, un pas d'écart suffit à tomber). Le détour sûr coûte `-17` (3 pas vers le haut, 11 vers la droite, 3 vers le bas). La différence entre les deux politiques est donc de 4 points de retour — et c'est précisément l'écart que Q-Learning et SARSA ne traitent pas de la même façon.

La raison est dans la cible de mise à jour. Q-Learning apprend `max_a Q(s', a)` : la valeur de la **meilleure** action à venir, comme si l'agent allait jouer optimalement ensuite. SARSA apprend `Q(s', a')` où `a'` est l'action **effectivement choisie** par la même politique `epsilon`-gloutonne — exploration comprise. Pendant l'apprentissage, cette exploration fait parfois tomber l'agent dans la falaise ; SARSA intègre ce risque dans ses valeurs et préfère le détour, alors que Q-Learning l'ignore et apprend le raccourci.


In [19]:
sealed class CliffWorld
{
    public const int Rows = 4, Cols = 12;
    public int StartState => (Rows - 1) * Cols;
    public int GoalState => Rows * Cols - 1;
    public int NStates => Rows * Cols;
    public int NActions => 4;
    // 0 = haut, 1 = droite, 2 = bas, 3 = gauche
    public readonly (int Dr, int Dc)[] Moves = { (-1, 0), (0, 1), (1, 0), (0, -1) };
    public readonly string[] ActionNames = { "H", "D", "B", "G" };

    public bool IsCliff(int s) => s / Cols == Rows - 1 && s % Cols >= 1 && s % Cols <= Cols - 2;
    public bool IsTerminal(int s) => s == GoalState || IsCliff(s);

    public (int Next, double Reward, bool Done) Step(int s, int a)
    {
        int r = s / Cols + Moves[a].Dr;
        int c = s % Cols + Moves[a].Dc;
        if (r < 0 || r >= Rows || c < 0 || c >= Cols) return (s, -1.0, false);
        int n = r * Cols + c;
        if (IsCliff(n)) return (StartState, -100.0, true);
        if (n == GoalState) return (n, -1.0, true);
        return (n, -1.0, false);
    }
}

var cliff = new CliffWorld();

// Un seul protocole pour les sections 7 a 9 : c'est la seule facon de comparer
// trois regles de mise a jour sans confondre l'effet de la regle avec celui des
// hyperparametres. epsilon decroit de 1 (exploration pure) vers 0.01.
const int CliffEpisodes = 10000;
const double CliffAlpha = 0.1;
const double CliffGamma = 0.99;
const double EpsStart = 1.0;
const double EpsEnd = 0.01;
const double EpsDecay = 0.999;
const int CliffMaxSteps = 200;

enum CliffRule { QLearning, Sarsa, ExpectedSarsa }

// E_pi[Q(s, .)] sous une politique epsilon-greedy : le max pese (1 - eps + eps/n),
// chaque action pese eps/n. C'est la seule ligne qui separe Expected SARSA de SARSA.
double ExpectedValue(double[,] Q, int state, double epsilon)
{
    int nA = Q.GetLength(1);
    double best = MaxAction(Q, state);
    double sum = 0;
    for (int a = 0; a < nA; a++) sum += Q[state, a];
    return (1.0 - epsilon + epsilon / nA) * best + (epsilon / nA) * sum;
}

// Trois algorithmes, une seule boucle : seule la CIBLE change.
// Ici toutes les valeurs sont negatives (chaque pas coute 1, la falaise 100).
// MaxAction est indispensable : un maximum amorce a zero plafonnerait le bootstrap
// a 0, rendrait toutes les actions artificiellement bonnes et detruirait l'ordre
// entre elles. Valeur positive par construction -> piege invisible.
(double[,] Q, double[] Returns) TrainCliff(CliffWorld env, CliffRule rule, int episodes,
    double alpha, double gamma, double epsStart, double epsEnd, double epsDecay, int seed,
    int maxSteps = CliffMaxSteps)
{
    var rng = new Random(seed);
    var Q = new double[env.NStates, env.NActions];
    var returns = new double[episodes];
    double eps = epsStart;

    for (int ep = 0; ep < episodes; ep++)
    {
        int s = env.StartState;
        int a = EpsilonGreedy(Q, s, eps, rng);
        double total = 0;

        for (int t = 0; t < maxSteps; t++)
        {
            var (next, reward, done) = env.Step(s, a);
            // Tiree une seule fois : c'est l'action que l'agent jouera au pas suivant,
            // et c'est elle que SARSA place dans sa cible (on-policy).
            int aNext = done ? 0 : EpsilonGreedy(Q, next, eps, rng);

            double target =
                done ? reward
                : rule == CliffRule.QLearning    ? reward + gamma * MaxAction(Q, next)
                : rule == CliffRule.Sarsa        ? reward + gamma * Q[next, aNext]
                :                                  reward + gamma * ExpectedValue(Q, next, eps);

            Q[s, a] += alpha * (target - Q[s, a]);
            total += reward;
            s = next;
            a = aNext;
            if (done) break;
        }

        eps = Math.Max(epsEnd, eps * epsDecay);
        returns[ep] = total;
    }
    return (Q, returns);
}

double GreedyReturn(CliffWorld env, double[,] q, int maxSteps = CliffMaxSteps)
{
    int s = env.StartState;
    double total = 0;
    for (int t = 0; t < maxSteps; t++)
    {
        var (n, r, done) = env.Step(s, GreedyAction(q, s));
        total += r;
        s = n;
        if (done) break;
    }
    return total;
}

// La case de depart porte son action a cote (SH, SD, ...) : c'est la case qui
// distingue les politiques, la masquer d'un simple "S" rendrait la grille muette
// sur la seule ligne qui compte.
string DrawCliffPolicy(CliffWorld env, double[,] q)
{
    var sb = new StringBuilder();
    for (int r = 0; r < CliffWorld.Rows; r++)
    {
        for (int c = 0; c < CliffWorld.Cols; c++)
        {
            int s = r * CliffWorld.Cols + c;
            string mark = s == env.GoalState ? "B"
                        : env.IsCliff(s) ? "#"
                        : s == env.StartState ? "S" + env.ActionNames[GreedyAction(q, s)]
                        : env.ActionNames[GreedyAction(q, s)];
            sb.Append(mark.PadLeft(3));
        }
        sb.Append('\n');
    }
    return sb.ToString();
}

In [20]:
var qlGreedy = new List<double>();
var sarsaGreedy = new List<double>();
var qlCurves = new List<double[]>();
var sarsaCurves = new List<double[]>();
var qlLast = new List<double>();
var sarsaLast = new List<double>();
double[,] qlQ = null, sarsaQ = null;

foreach (var seed in SweepSeeds)
{
    var (qQl, rQl) = TrainCliff(cliff, CliffRule.QLearning, CliffEpisodes, CliffAlpha, CliffGamma, EpsStart, EpsEnd, EpsDecay, seed);
    var (qSarsa, rSarsa) = TrainCliff(cliff, CliffRule.Sarsa, CliffEpisodes, CliffAlpha, CliffGamma, EpsStart, EpsEnd, EpsDecay, seed);

    qlCurves.Add(MovingAverage(rQl, 20));
    sarsaCurves.Add(MovingAverage(rSarsa, 20));
    qlGreedy.Add(GreedyReturn(cliff, qQl));
    sarsaGreedy.Add(GreedyReturn(cliff, qSarsa));
    qlLast.Add(rQl.TakeLast(200).Average());
    sarsaLast.Add(rSarsa.TakeLast(200).Average());
    qlQ = qQl; sarsaQ = qSarsa;

    Console.WriteLine($"graine {seed,2} : Q-Learning retour glouton {qlGreedy[^1],5:0}   |   SARSA {sarsaGreedy[^1],5:0}   |   retour d'apprentissage des 200 derniers episodes : QL {qlLast[^1],7:0.0} / SARSA {sarsaLast[^1],7:0.0}");
}

string Span(List<double> v) => $"{v.Average(),6:0.0}  [{v.Min(),5:0} .. {v.Max(),5:0}]";

double[] MeanCurve(List<double[]> curves)
{
    int n = curves.Min(c => c.Length);
    return Enumerable.Range(0, n).Select(i => curves.Average(c => c[i])).ToArray();
}

Console.WriteLine();
Console.WriteLine($"Retour glouton sur 4 graines — Q-Learning : {Span(qlGreedy)}   |   SARSA : {Span(sarsaGreedy)}");
Console.WriteLine($"Retour d'apprentissage      — Q-Learning : {Span(qlLast)}   |   SARSA : {Span(sarsaLast)}");
Console.WriteLine("(optimum -13 : 13 pas sans toucher la falaise ; detour d'une rangee de plus : -17)");

var meanQl = MeanCurve(qlCurves);
var meanSarsa = MeanCurve(sarsaCurves);
var xs = new List<double>();
var ysQl = new List<double>();
var ysSarsa = new List<double>();
for (int i = 0; i < meanQl.Length; i += 100)
{
    xs.Add(i + 1);
    ysQl.Add(meanQl[i]);
    ysSarsa.Add(meanSarsa[i]);
}

var cliffSeries = new List<(string Label, double[] Xs, double[] Ys, string Color)>
{
    ("Q-Learning", xs.ToArray(), ysQl.ToArray(), "#1f77b4"),
    ("SARSA", xs.ToArray(), ysSarsa.ToArray(), "#d62728"),
};

PlotLines("CliffWalking : retour par episode (moyenne de 4 graines, fenetre 20, un point sur 100)",
          "episode", "retour de l'episode", cliffSeries).Display();

Console.WriteLine();
Console.WriteLine("Politique gloutonne — Q-Learning :");
Console.WriteLine(DrawCliffPolicy(cliff, qlQ));
Console.WriteLine("Politique gloutonne — SARSA :");
Console.WriteLine(DrawCliffPolicy(cliff, sarsaQ));

graine  0 : Q-Learning retour glouton   -13   |   SARSA   -17   |   retour d'apprentissage des 200 derniers episodes : QL   -14.0 / SARSA   -17.6


graine  1 : Q-Learning retour glouton   -13   |   SARSA   -17   |   retour d'apprentissage des 200 derniers episodes : QL   -15.5 / SARSA   -17.2


graine  7 : Q-Learning retour glouton   -13   |   SARSA   -17   |   retour d'apprentissage des 200 derniers episodes : QL   -14.6 / SARSA   -17.6


graine 42 : Q-Learning retour glouton   -13   |   SARSA   -17   |   retour d'apprentissage des 200 derniers episodes : QL   -15.5 / SARSA   -17.6


Retour glouton sur 4 graines — Q-Learning :  -13.0  [  -13 ..   -13]   |   SARSA :  -17.0  [  -17 ..   -17]


Retour d'apprentissage      — Q-Learning :  -14.9  [  -16 ..   -14]   |   SARSA :  -17.5  [  -18 ..   -17]


... (2 outputs left out)

Politique gloutonne — Q-Learning :


  D  B  G  D  D  D  G  D  D  D  B  B
  D  D  D  D  D  D  D  B  D  B  D  B
  D  D  D  D  D  D  D  D  D  D  D  B
 SH  #  #  #  #  #  #  #  #  #  #  B



Politique gloutonne — SARSA :


  D  D  D  D  D  D  D  D  D  D  D  B
  H  H  H  D  H  H  H  D  D  D  D  B
  H  H  H  H  H  H  H  H  H  H  D  B
 SH  #  #  #  #  #  #  #  #  #  #  B



### Lecture du résultat

Mêmes hyperparamètres, même environnement, même calendrier d'exploration : **Q-Learning rend −13,0 et SARSA −17,0, sur les quatre graines, sans dispersion** (`[-13 .. -13]` et `[-17 .. -17]`). L'écart n'est donc pas du bruit d'échantillonnage, c'est un désaccord entre deux définitions de « la valeur d'une action ».

**Les deux trajectoires, lisibles dans les grilles.** Toutes deux partent par `H` — le départ est en bas à gauche, il faut d'abord monter. Q-Learning monte d'une rangée puis file à droite sur la rangée 2, et redescend au but : `1 + 11 + 1 = 13` pas, soit exactement l'optimum. SARSA monte de **deux** rangées, traverse la rangée 0, puis redescend en trois pas : `3 + 11 + 3 = 17`. La différence tient à une seule rangée d'écart, et c'est tout le sujet : la rangée 2 longe la falaise, la rangée 0 s'en éloigne.

**Pourquoi une seule rangée change le résultat.** La cible de Q-Learning est $\max_{a'} Q(s',a')$ : elle évalue chaque état *comme si* l'action gloutonne y était toujours jouée. Cette hypothèse est fausse pendant l'apprentissage — un pas sur dix (puis un sur cent) est tiré au hasard, et un pas aléatoire vers le bas depuis la rangée 2 tombe dans la falaise pour −100. Q-Learning ignore ce risque : il price l'agent parfait, et l'agent parfait longe le bord. La cible de SARSA est $Q(s',a')$ avec $a'$ l'action **effectivement tirée** : le risque d'exploration entre dans la valeur, et la politique apprise prend une rangée de marge. Les −17 sont le prix de cette marge, pas une contre-performance.

**Retour glouton et retour d'apprentissage ne mesurent pas la même chose.** Le retour glouton (−13 / −17) évalue la politique finale **sans exploration** : c'est ce que l'agent sait. Le retour des 200 derniers épisodes (−14,9 / −17,5) est mesuré **pendant** l'apprentissage, avec l'exploration résiduelle. Ici les deux se suivent de près, et pour une raison mécanique : $\varepsilon$ décroît de 1,0 à `0.01` en `1.0 * 0.999^10000 ≈ 4.5e-5`, donc il est déjà au plancher depuis l'épisode ~4 600. La sonde du régime classique — $\varepsilon$ maintenu à 0,1, où SARSA redevient meilleur **en ligne** et Q-Learning meilleur **glouton** — est l'exercice 3.

**Le piège qui a produit −200.** La première version de `TrainCliff` amorçait son maximum à zéro (`double best = 0;` puis `if (Q[n, k] > best)`). Sur FrozenLake, toutes les valeurs sont positives : le maximum est correct et le défaut invisible. Ici toutes les valeurs sont négatives, donc le maximum reste **collé à zéro**, toutes les actions paraissent également bonnes, et la politique apprise ne trouve plus le but — retour −200 sur les quatre graines, c'est-à-dire l'épuisement des 200 pas. `MaxAction` amorce sur `Q[s, 0]`, une valeur de la table. Retenir la règle : **un accumulateur de maximum ne s'amorce jamais sur zéro sans preuve que les valeurs sont positives.**


## 8. Expected SARSA : la moyenne plutôt que l'échantillon

Q-Learning prend le **maximum** des valeurs de l'état suivant, SARSA prend la valeur de **l'action tirée**. Expected SARSA prend l'**espérance** de ces valeurs sous la politique comportementale :

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \sum_{a'} \pi(a' \mid s')\, Q(s',a') - Q(s,a) \right]
$$

Sous une politique $\varepsilon$-greedy à $n$ actions, cette espérance s'écrit sans somme sur la politique :

$$
\sum_{a'} \pi(a' \mid s')\, Q(s',a') \;=\; \left(1-\varepsilon+\frac{\varepsilon}{n}\right) \max_{a'} Q(s',a') \;+\; \frac{\varepsilon}{n} \sum_{a'} Q(s',a')
$$

Le terme du maximum pèse $1-\varepsilon$, chaque action pèse $\varepsilon/n$ : c'est la même politique que SARSA, mais **intégrée au lieu d'être échantillonnée**.

**Ce que cela change.** La cible ne dépend plus d'un tirage : même signal moyen, variance d'échantillonnage en moins. Le coût est un balayage de la ligne $Q(s', \cdot)$ à chaque mise à jour — quatre entrées ici, donc négligeable en tabulaire, mais un passage complet dans le réseau une fois la valeur approchée.

La place attendue entre les deux extrêmes est directe : le maximum suppose l'agent parfait, l'échantillon paie chaque incertitude de tirage, l'espérance est la moyenne exacte sous la politique réellement suivie.


In [21]:
var esGreedy = new List<double>();
var esLast = new List<double>();
var esCurves = new List<double[]>();
double[,] esQ = null;

foreach (var seed in SweepSeeds)
{
    var (qEs, rEs) = TrainCliff(cliff, CliffRule.ExpectedSarsa, CliffEpisodes, CliffAlpha, CliffGamma, EpsStart, EpsEnd, EpsDecay, seed);
    esGreedy.Add(GreedyReturn(cliff, qEs));
    esLast.Add(rEs.TakeLast(200).Average());
    esCurves.Add(MovingAverage(rEs, 20));
    esQ = qEs;
    Console.WriteLine($"graine {seed,2} : Expected SARSA retour glouton {esGreedy[^1],5:0}   |   retour d'apprentissage des 200 derniers episodes : {esLast[^1],7:0.0}");
}

string Triplet(List<double> v) => $"{v.Average(),6:0.0}  [{v.Min(),5:0} .. {v.Max(),5:0}]";
string Trail(double greedyReturn) => Math.Abs(greedyReturn) < CliffMaxSteps
    ? $"{-(int)greedyReturn,3} pas"
    : "bloque";

Console.WriteLine();
Console.WriteLine("regle            | cible                                | retour glouton        | chemin  | apprentissage (200 derniers)");
Console.WriteLine("-----------------+--------------------------------------+-----------------------+---------+------------------------------");
Console.WriteLine($"Q-Learning       | max_a' Q(s', a')                     | {Triplet(qlGreedy)} | {Trail(qlGreedy.Average())}   | {Triplet(qlLast)}");
Console.WriteLine($"Expected SARSA   | E_pi[Q(s', .)] sous eps-greedy       | {Triplet(esGreedy)} | {Trail(esGreedy.Average())}   | {Triplet(esLast)}");
Console.WriteLine($"SARSA            | Q(s', a') avec a' tire par pi        | {Triplet(sarsaGreedy)} | {Trail(sarsaGreedy.Average())}   | {Triplet(sarsaLast)}");

var meanEs = MeanCurve(esCurves);
var xs3 = new List<double>();
var yQl = new List<double>();
var yEs = new List<double>();
var ySarsa = new List<double>();
var meanQl3 = MeanCurve(qlCurves);
var meanSarsa3 = MeanCurve(sarsaCurves);
for (int i = 0; i < meanEs.Length; i += 100)
{
    xs3.Add(i + 1);
    yQl.Add(meanQl3[i]);
    yEs.Add(meanEs[i]);
    ySarsa.Add(meanSarsa3[i]);
}

var threeSeries = new List<(string Label, double[] Xs, double[] Ys, string Color)>
{
    ("Q-Learning (max)", xs3.ToArray(), yQl.ToArray(), "#1f77b4"),
    ("Expected SARSA (esperance)", xs3.ToArray(), yEs.ToArray(), "#2ca02c"),
    ("SARSA (echantillon)", xs3.ToArray(), ySarsa.ToArray(), "#d62728"),
};

PlotLines("CliffWalking : les trois cibles de mise a jour (moyenne de 4 graines, fenetre 20, un point sur 100)",
          "episode", "retour de l'episode", threeSeries).Display();

Console.WriteLine();
Console.WriteLine("Politique gloutonne — Expected SARSA :");
Console.WriteLine(DrawCliffPolicy(cliff, esQ));

graine  0 : Expected SARSA retour glouton   -15   |   retour d'apprentissage des 200 derniers episodes :   -15.2


graine  1 : Expected SARSA retour glouton   -15   |   retour d'apprentissage des 200 derniers episodes :   -15.2


graine  7 : Expected SARSA retour glouton   -15   |   retour d'apprentissage des 200 derniers episodes :   -15.2


graine 42 : Expected SARSA retour glouton   -15   |   retour d'apprentissage des 200 derniers episodes :   -15.1


regle            | cible                                | retour glouton        | chemin  | apprentissage (200 derniers)


-----------------+--------------------------------------+-----------------------+---------+------------------------------


... (2 outputs left out)

SARSA            | Q(s', a') avec a' tire par pi        |  -17.0  [  -17 ..   -17] |  17 pas   |  -17.5  [  -18 ..   -17]


CliffWalking : les trois cibles de mise a jour (moyenne de 4 graines, fenetre 20, un point sur 100) -119.7295 -91.6648 -63.6 -35.5353 -7.4705 1 81 161 241 321 401 episode retour de l'episode Q-Learning (max) Expected SARSA (esperance) SARSA (echantillon)

Politique gloutonne — Expected SARSA :


  D  D  D  D  D  D  D  D  D  D  D  B
  D  D  D  D  D  D  D  D  D  D  D  B
  H  H  H  H  H  H  H  H  H  D  D  B
 SH  #  #  #  #  #  #  #  #  #  #  B



### Lecture du résultat

Les trois règles, mêmes hyperparamètres, mêmes quatre graines :

| règle | cible | retour glouton | chemin | apprentissage (200 derniers) |
|---|---|---|---|---|
| Q-Learning | $\max_{a'} Q(s',a')$ | **−13,0** `[-13 .. -13]` | 13 pas | −14,9 `[-16 .. -14]` |
| Expected SARSA | $\mathbb{E}_\pi[Q(s',\cdot)]$ | **−15,0** `[-15 .. -15]` | 15 pas | −15,2 `[-15 .. -15]` |
| SARSA | $Q(s',a')$, $a'$ tiré | **−17,0** `[-17 .. -17]` | 17 pas | −17,5 `[-18 .. -17]` |

**Une échelle, pas un classement.** 13, 15, 17 pas : trois chemins de longueur croissante, et une seule différence de code entre eux — la cible. Q-Learning longe la falaise d'une rangée au-dessus, Expected SARSA passe par la rangée du milieu, SARSA monte jusqu'à la rangée du haut. Chaque rangée gagnée vers le haut coûte exactement 2 pas de plus. Aucune graine ne s'écarte du résultat : à 10 000 épisodes et $\varepsilon$ décroissant, les trois tables convergent vers trois points fixes distincts, pas vers trois tirages d'un même point fixe.

**La variance d'échantillonnage se voit dans la dispersion.** C'est la colonne qui valide la théorie : l'apprentissage d'Expected SARSA tient dans `[-15 .. -15]`, soit une amplitude **nulle** sur 4 graines ; Q-Learning s'étale sur `[-16 .. -14]`, SARSA sur `[-18 .. -17]`. Les deux autres tirent leur cible — le premier un maximum sur une table encore instable, le second une action au hasard ; Expected SARSA intègre, donc sa cible ne dépend plus d'un tirage. Même signal moyen, bruit en moins : c'est exactement ce que promettait la formule, et c'est mesuré.

**Pourquoi la politique du milieu reste stable alors que $\varepsilon$ finit à 0,01.** À la fin, l'espérance ne pèse plus que 1 % au-delà du maximum, et l'on pourrait croire que la cible rejoint celle de Q-Learning. Elle ne la rejoint pas *exactement*, et ce pour cent suffit : la falaise coûte 100, donc la petite fraction de probabilité qui pointe vers le bas depuis la rangée 2 reste chère en proportion, et cette rangée reste défavorisée. La cible est un continuum : à \(\varepsilon = 1\) c'est la moyenne de tout, à \(\varepsilon = 0\) c'est le maximum. CliffWalking reste sur la trajectoire intermédiaire parce qu'il paie le risque pendant tout l'apprentissage, avant que \(\varepsilon\) ne s'effondre.

**Ce que cela coûte.** Une passe sur les $n$ actions de la ligne à chaque mise à jour. Ici $n = 4$ et le coût est invisible ; sur une valeur approchée par un réseau, il faut évaluer toutes les actions de l'état suivant à chaque pas, ce qui devient le facteur limitant. C'est le seul argument qui fait préférer SARSA, et il n'a rien à voir avec la qualité de la politique apprise.


## 9. n-step TD : le compromis biais-variance

SARSA et Q-Learning sont des méthodes **TD(0)** : la cible utilise une récompense réelle et une seule valeur estimée. Les méthodes **n-step** généralisent en bootstrapant après $n$ récompenses réellement reçues :

$$
G_t^{(n)} = r_{t+1} + \gamma\, r_{t+2} + \dots + \gamma^{n-1} r_{t+n} + \gamma^n Q(s_{t+n}, a_{t+n})
$$

Deux cas limites encadrent la famille :

- $n = 1$ : purement bootstrap — biais fort (on fait confiance à une valeur encore fausse), variance faible.
- $n \to \infty$ : Monte-Carlo — biais nul, variance maximale (la cible dépend de toute la trajectoire, exploration comprise).

Le paramètre $n$ déplace donc la mise à jour sur un curseur unique : **combien de récompenses réelles avant de faire confiance à l'estimation**. L'ablation qui suit rejoue SARSA avec $n \in \{1, 3, 5\}$ sur les mêmes quatre graines — mêmes initialisations, même calendrier d'exploration — de façon à comparer $n$ à graines appariées. Les trajectoires, elles, divergent dès que $n$ change les mises à jour (donc la politique comportementale, donc la consommation du générateur) : la comparaison garantit des départs appariés, pas des épisodes identiques.


In [22]:
// Applique une mise a jour n-step sur l'etat d'indice tau.
// bootstrapIndex = -1 quand l'episode est fini : la fenetre est alors tronquee par
// le terminal et l'on n'a plus d'etat a evaluer. Positif, c'est l'indice de l'etat
// atteint n pas plus tard, celui dont la valeur amorce la cible.
// greedyBootstrap = false : on amorce sur Q(s_{t+n}, a_{t+n}), l'action REELLEMENT jouee
//   par la politique de comportement -> n-step SARSA, on-politique.
// greedyBootstrap = true : on amorce sur max_a Q(s_{t+n}, a), independamment de l'action
//   jouee -> n-step hors-politique.
void NStepUpdate(double[,] Q, List<int> states, List<int> actions, List<double> rewards,
                 int tau, int n, int bootstrapIndex, bool greedyBootstrap,
                 double alpha, double gamma)
{
    double target = 0;
    int horizon = Math.Min(n, rewards.Count - tau);
    for (int k = 0; k < horizon; k++) target += Math.Pow(gamma, k) * rewards[tau + k];

    if (bootstrapIndex > 0)
    {
        int sB = states[bootstrapIndex];
        double qB = greedyBootstrap ? MaxAction(Q, sB) : Q[sB, actions[bootstrapIndex]];
        target += Math.Pow(gamma, n) * qB;
    }

    int s0 = states[tau], a0 = actions[tau];
    Q[s0, a0] += alpha * (target - Q[s0, a0]);
}

// n-step : la cible additionne n recompenses reelles puis bootstrappe.
// n = 1 redonne exactement la methode TD(0) correspondante ; n grand tend vers Monte-Carlo.
(double[,] Q, double[] Returns) TrainNStep(CliffWorld env, int n, bool greedyBootstrap, int episodes,
    double alpha, double gamma, double epsStart, double epsEnd, double epsDecay, int seed,
    int maxSteps = CliffMaxSteps)
{
    var rng = new Random(seed);
    var Q = new double[env.NStates, env.NActions];
    var returns = new double[episodes];
    double eps = epsStart;

    for (int ep = 0; ep < episodes; ep++)
    {
        var states = new List<int> { env.StartState };
        var actions = new List<int> { EpsilonGreedy(Q, env.StartState, eps, rng) };
        var rewards = new List<double>();
        double total = 0;
        int t = 0;
        bool done = false;

        while (!done && t < maxSteps)
        {
            var (next, reward, d) = env.Step(states[t], actions[t]);
            rewards.Add(reward);
            total += reward;
            states.Add(next);
            actions.Add(d ? 0 : EpsilonGreedy(Q, next, eps, rng));
            t++;
            done = d;

            // La cible de l'etat t-n est complete des que le pas t est joue.
            int tau = t - n;
            if (tau >= 0)
                NStepUpdate(Q, states, actions, rewards, tau, n, done ? -1 : t, greedyBootstrap, alpha, gamma);
        }

        // Fin d'episode : les n-1 derniers pas, tronques par le terminal.
        for (int tau = Math.Max(0, t - n + 1); tau < t; tau++)
            NStepUpdate(Q, states, actions, rewards, tau, n, -1, greedyBootstrap, alpha, gamma);

        eps = Math.Max(epsEnd, eps * epsDecay);
        returns[ep] = total;
    }
    return (Q, returns);
}

double Median(List<double> v)
{
    var s = v.OrderBy(x => x).ToArray();
    int m = s.Length / 2;
    return s.Length % 2 == 1 ? s[m] : (s[m - 1] + s[m]) / 2.0;
}

double Std(List<double> v)
{
    double m = v.Average();
    return Math.Sqrt(v.Sum(x => (x - m) * (x - m)) / v.Count);
}

// Deux amorcages, un controleur : a n = 1, la colonne de gauche doit redonner le
// SARSA de la section 7 et celle de droite le Q-Learning de la section 7.
int[] horizons = { 1, 3, 5 };
var nStepResults = new Dictionary<(int N, bool Greedy), List<double>>();
var nStepLearning = new Dictionary<(int N, bool Greedy), double>();

Console.WriteLine(" n | n-step SARSA (action jouee)      | n-step Q (amorcage glouton)");
Console.WriteLine("---+----------------------------------+----------------------------------");
foreach (int n in horizons)
{
    var cells = new List<string>();
    foreach (bool greedy in new[] { false, true })
    {
        var vals = new List<double>();
        var late = new List<double>();
        foreach (var seed in SweepSeeds)
        {
            var (qN, rets) = TrainNStep(cliff, n, greedy, CliffEpisodes, CliffAlpha, CliffGamma,
                                        EpsStart, EpsEnd, EpsDecay, seed);
            vals.Add(GreedyReturn(cliff, qN));
            late.Add(rets.Skip(CliffEpisodes - 200).Average());
        }
        nStepResults[(n, greedy)] = vals;
        nStepLearning[(n, greedy)] = late.Average();
        cells.Add($"{Median(vals),6:0.0} [{vals.Min(),4:0} ..{vals.Max(),4:0}] std {Std(vals),4:0.00}");
    }
    Console.WriteLine($"{n,2} | {cells[0]} | {cells[1]}");
}

Console.WriteLine();
Console.WriteLine("Retour moyen des 200 derniers episodes (pendant l'apprentissage)");
Console.WriteLine(" n | n-step SARSA | n-step Q");
Console.WriteLine("---+--------------+---------");
foreach (int n in horizons)
    Console.WriteLine($"{n,2} | {nStepLearning[(n, false)],12:0.0} | {nStepLearning[(n, true)],8:0.0}");

 n | n-step SARSA (action jouee)      | n-step Q (amorcage glouton)


---+----------------------------------+----------------------------------


 1 |  -17.0 [ -17 .. -17] std 0.00 |  -13.0 [ -13 .. -13] std 0.00


 3 |  -17.0 [ -17 .. -17] std 0.00 |  -17.0 [ -17 .. -17] std 0.00


 5 |  -17.0 [ -17 .. -17] std 0.00 |  -17.0 [ -17 .. -17] std 0.00


Retour moyen des 200 derniers episodes (pendant l'apprentissage)


 n | n-step SARSA | n-step Q


---+--------------+---------


 1 |        -17.5 |    -14.9


 3 |        -17.3 |    -17.5


 5 |        -17.3 |    -17.4


### Lecture de l'ablation n-step

La matrice porte **deux contrôles croisés**, et c'est eux qui rendent la mesure falsifiable. À `n = 1`, la cible n-step ne contient plus qu'une récompense réelle suivie d'un terme d'amorçage : elle **doit** reproduire exactement la méthode TD(0) correspondante, sur les mêmes graines et le même protocole.

| variante à `n = 1` | retour glouton | section 7 | apprentissage (200 derniers) | section 7 |
|---|---|---|---|---|
| amorçage sur l'action jouée (SARSA) | −17.0 `[-17 .. -17]` | SARSA −17.0 | −17.5 | −17.5 |
| amorçage glouton (hors-politique) | −13.0 `[-13 .. -13]` | Q-Learning −13.0 | −14.9 | −14.9 |

Quatre valeurs, quatre concordances au dixième près. Ce recoupement attrape une erreur de fenêtre **avant** qu'elle ne se voie ailleurs : si le garde d'amorçage interdit le dernier pas de la fenêtre, plus aucune cible ne reçoit de terme bootstrap, la valeur d'un état se réduit à ses récompenses immédiates, et les trois horizons rendent le plafond `CliffMaxSteps` (−200) — y compris `n = 1`, qui est précisément le cas où le contrôle sait ce qu'il devrait lire.

**Ce que l'horizon ne change pas.** Dès `n ≥ 3`, les deux variantes convergent vers le **même retour glouton −17.0**, dispersion nulle sur les quatre graines. Avec ε maintenu à 0.01 sur les derniers milliers d'épisodes, la politique gloutonne finale ne dépend plus de l'horizon : c'est le chemin sûr du milieu. L'horizon ne déplace donc pas l'optimum atteint sur ce MDP, il déplace la **trajectoire** — et peu : le retour des 200 derniers épisodes vaut −17.5, −17.3, −17.3 pour le SARSA n-step.

**Ce que l'horizon change, et ce qui n'est pas l'horizon.** Le seul écart qui subsiste entre les deux colonnes est le `n = 1` hors-politique, et il ne vient pas du nombre de récompenses réelles : il vient de la **règle d'amorçage**. Amorcer sur `max_a Q(s', a)` plutôt que sur l'action jouée suffit à faire prendre le chemin du bord (13 pas, −13) au lieu du détour (17 pas, −17). C'est l'arbitrage de la section 7 retrouvé par un autre chemin : la forme de la cible pèse plus que l'horizon qui la remplit.

**Le compromis biais-variance n'est pas visible ici, et cela se dit.** Le manuel annonce que `n → ∞` fait croître la variance et décroître le biais. Sur ce protocole, `n = 5` ne coûte ni une graine ni un dixième de retour. Deux raisons mesurables : les épisodes de CliffWalking sont courts (17 pas en glouton, quelques dizaines en exploration), donc `n = 5` reste très loin de Monte-Carlo ; et ε figé à 0.01 sur 10 000 épisodes laisse la politique de comportement presque déterministe, donc l'échantillon de trajectoires très concentré. Le versant variance demanderait des épisodes longs à récompense rare — un couloir, un labyrinthe à clé — hors du cadre lisible de cette grille.

**Écart avec le jumeau Python, nommé plutôt que tu.** Le carnet Python appelle sa fonction `n_step_sarsa` et sa docstring annonce « pour n=1, équivalent a SARSA classique », mais son amorçage est `Q[next_state, np.argmax(Q[next_state])]` : il n'utilise jamais l'action effectivement jouée. C'est donc la colonne de **droite** que ce carnet calcule, et c'est pourquoi il rapporte −13 à `n = 1` — la valeur de Q-Learning, pas celle de SARSA. Le jumeau C# livre les deux colonnes explicitement, pour que le lecteur voie l'arbitrage au lieu de le subir. Pour `n = 5`, le carnet Python rapporte une dispersion de 0.87 avec une graine à −19 ; elle ne se reproduit pas ici sur les mêmes quatre graines. Les deux implémentations diffèrent sur le moment où la fenêtre se met à jour (son garde `len(states) > n` déclenche la mise à jour un pas plus tard que `tau = t - n`) et sur la construction du vidage final ; laquelle de ces deux différences produit le −19 n'est pas établi par cette mesure.

## 10. Comparaison des méthodes

Six algorithmes ont tourné dans ce carnet, sur deux environnements et un seul protocole par famille. La comparaison utile n'est pas un classement — elle dépend du régime — mais une lecture de **ce que chaque méthode suppose et de ce qu'elle écrit comme valeur**. La cellule suivante recompose le tableau à partir des objets déjà calculés : aucune valeur n'y est recopiée à la main, donc le tableau se réexécute avec le carnet et ne peut pas dériver de ses propres mesures.

In [23]:
// Tout ce qui suit est lu dans les variables deja calculees par les sections 3 a 9.
// Le tableau se reexecute avec le carnet : il ne peut pas deriver de ses mesures.
int sweepsPi = historyPi.Sum(h => h.Sweeps);
int sweepsPiSlip = historyPiSlip.Sum(h => h.Sweeps);

// Largeur fixe par construction : une colonne qui deborde d'un caractere deforme
// toutes les lignes suivantes, et l'oeil ne le rattrape pas.
// Les libelles sont hisses dans des variables : un guillemet echappe a l'interieur
// d'un trou d'interpolation est une erreur de compilation, pas une chaine.
const int Col = 22;
string Name(string s) => s.PadRight(Col);
string HdMethod = Name("methode");
string Sep1 = new string('-', Col) + "+-----------+------------+--------------------------------";
string Sep2 = new string('-', Col) + "+---------------------------------+---------------------------+------------------------";
string LblVi = Name("Iteration de Valeur");
string LblPi = Name("Iteration de Politique");
string LblQl = Name("Q-Learning (1 pas)");
string LblEs = Name("Expected SARSA");
string LblSarsa = Name("SARSA (1 pas)");
string LblN1 = Name("n-step SARSA, n = 1");
string LblN3 = Name("n-step SARSA, n = 3");
string LblN5 = Name("n-step SARSA, n = 5");
string LblNg1 = Name("n-step glouton, n = 1");

// nStepLearning ne stocke qu'une moyenne sur les 4 graines : lui donner la forme
// d'un intervalle [min .. max] afficherait une dispersion qui n'a pas ete mesuree.
string AvgOf(double v) => $"{v,7:0.0}  moy. 4 graines";

Console.WriteLine("PLANIFICATION -- le modele P est connu, aucune interaction");
Console.WriteLine($"{HdMethod}| balayages | V*(depart) | modele");
Console.WriteLine(Sep1);
Console.WriteLine($"{LblVi}| {deltasVi.Count,9} | {vStar[frozen.StartState],10:0.000} | non glissant, delta tombe a zero");
Console.WriteLine($"{LblVi}| {deltasSlip.Count,9} | {vStarSlip[frozenSlippery.StartState],10:0.000} | glissant, 3 issues par action");
Console.WriteLine($"{LblPi}| {sweepsPi,9} | {vPi[frozen.StartState],10:0.000} | non glissant, {historyPi.Count} ameliorations");
Console.WriteLine($"{LblPi}| {sweepsPiSlip,9} | {vPiSlip[frozenSlippery.StartState],10:0.000} | glissant, {historyPiSlip.Count} ameliorations");

Console.WriteLine();
Console.WriteLine("APPRENTISSAGE -- le modele est inconnu, seuls des quintuplets sont observes");
Console.WriteLine($"{HdMethod}| cible de mise a jour            | retour glouton sur 4 graines | en ligne (200 derniers)");
Console.WriteLine(Sep2);
Console.WriteLine($"{LblQl}| max_a' Q(s', a')                | {Triplet(qlGreedy)} | {Triplet(qlLast)}");
Console.WriteLine($"{LblEs}| E_pi[Q(s', .)]                  | {Triplet(esGreedy)} | {Triplet(esLast)}");
Console.WriteLine($"{LblSarsa}| Q(s', a') avec a' jouee          | {Triplet(sarsaGreedy)} | {Triplet(sarsaLast)}");
Console.WriteLine($"{LblN1}| Q(s_1, a_1)                     | {Triplet(nStepResults[(1, false)])} | {AvgOf(nStepLearning[(1, false)])}");
Console.WriteLine($"{LblN3}| r + g r' + g2 r'' + g3 Q(s_3)   | {Triplet(nStepResults[(3, false)])} | {AvgOf(nStepLearning[(3, false)])}");
Console.WriteLine($"{LblN5}| 5 recompenses puis bootstrap    | {Triplet(nStepResults[(5, false)])} | {AvgOf(nStepLearning[(5, false)])}");
Console.WriteLine($"{LblNg1}| max_a' Q(s', a') (off-policy)   | {Triplet(nStepResults[(1, true)])} | {AvgOf(nStepLearning[(1, true)])}");

Console.WriteLine();
Console.WriteLine("Les deux lignes a n = 1 recoupent les sections 7 et 9 :");
Console.WriteLine($"  n-step SARSA n = 1    {Triplet(nStepResults[(1, false)])}  contre SARSA 1 pas   {Triplet(sarsaGreedy)}");
Console.WriteLine($"  n-step glouton n = 1  {Triplet(nStepResults[(1, true)])}  contre Q-Learning    {Triplet(qlGreedy)}");

PLANIFICATION -- le modele P est connu, aucune interaction


methode               | balayages | V*(depart) | modele


----------------------+-----------+------------+--------------------------------


Iteration de Valeur   |         7 |      0.951 | non glissant, delta tombe a zero


Iteration de Valeur   |       324 |      0.542 | glissant, 3 issues par action


Iteration de Politique|        13 |      0.951 | non glissant, 7 ameliorations


... (10 outputs left out)

n-step glouton, n = 1 | max_a' Q(s', a') (off-policy)   |  -13.0  [  -13 ..   -13] |   -14.9  moy. 4 graines


Les deux lignes a n = 1 recoupent les sections 7 et 9 :


  n-step SARSA n = 1     -17.0  [  -17 ..   -17]  contre SARSA 1 pas    -17.0  [  -17 ..   -17]


  n-step glouton n = 1   -13.0  [  -13 ..   -13]  contre Q-Learning     -13.0  [  -13 ..   -13]


### Lecture du résultat

Chaque colonne du tableau répond à une question différente, et c'est pourquoi il faut le lire colonne par colonne plutôt que ligne par ligne.

**Ce que la méthode doit savoir avant de commencer.** Une seule frontière traverse les six lignes : deux méthodes lisent `P`, quatre n'ont vu que des transitions. Cette frontière se voit dans la forme des résultats avant de se voir dans les colonnes — les quatre lignes de planification n'ont pas de graine, leur résultat est exact et reproductible au bit près ; les six lignes d'apprentissage portent un intervalle sur quatre graines, parce qu'elles dépendent de trajectoires échantillonnées. Passer de l'une à l'autre n'est pas changer d'algorithme, c'est changer de **contrat** : on échange l'accès au modèle contre l'accès aux trajectoires.

**Ce que la méthode écrit comme valeur.** `V*(s)` pour l'itération de valeur, `V^\pi(s)` pour l'itération de politique — la valeur de *sa* politique, pas de l'optimale —, puis `Q` sous trois définitions concurrentes de la cible. La colonne « cible de mise à jour » est exactement ce qui change d'une règle TD à l'autre : la boucle, le calendrier d'exploration, la table et les graines sont identiques. Deux méthodes qui ne diffèrent que par cette ligne produisent deux politiques différentes, et c'est le résultat central du carnet.

**Ce que la cible coûte en performance.** Sur CliffWalking, quatre cibles, quatre chemins : 13, 15 et 17 pas, et chaque rangée gagnée vers le haut coûte exactement deux pas. L'ordre suit l'hypothèse que chaque cible fait sur la politique future — l'agent parfait (`max`), la moyenne exacte sous la politique suivie (`espérance`), un tirage réel de cette politique (`échantillon`).

**Ce que le protocole ne montre pas, et qu'il ne faut pas lui faire dire.** Sur ce carnet, le retour *en ligne* classe les règles dans le même ordre que le retour glouton (−14,9 / −15,2 / −17,5). Le résultat classique — SARSA meilleur **pendant** l'apprentissage, Q-Learning meilleur **glouton** — demande un $\varepsilon$ qui ne s'effondre pas ; ici $\varepsilon$ atteint son plancher de 0,01 dès l'épisode ~4 600, donc la politique mesurée en ligne est déjà presque la politique gloutonne. Cette expérience est l'exercice 3.

**Ce que chaque méthode paie en calcul.** L'itération de valeur balaie tout le modèle par itération ; l'itération de politique résout une politique complète avant d'en changer une action, et sur ces deux grilles elle est **plus chère** en balayages (13 et 976 contre 7 et 324) — l'avantage classique de « peu de cycles » n'apparaît qu'avec beaucoup d'états. Expected SARSA balaie la ligne d'actions de l'état suivant à chaque mise à jour. Le n-step doit garder une fenêtre de `n` transitions et ne peut rien apprendre avant le `n`-ième pas d'un épisode.

**Ce que ce tableau n'est pas.** Ce n'est pas un classement général : 16 états pour FrozenLake, 48 pour CliffWalking, aucune approximation de fonction, aucun état continu. Sur ces tailles, un algorithme plus « avancé » ne gagne rien de mesurable ici — ce que le tableau compare, c'est la **définition de la valeur**, pas la puissance des méthodes.

## 11. Exercices

Sept exercices, du réglage d'hyperparamètre à l'algorithme à traces. Chacun dispose d'une cellule d'espace de travail qui **s'exécute** : elle déclare les variables à remplir et affiche l'état « à compléter », sans lever d'exception et sans contenir la solution. Les indicateurs donnent la piste, pas le code.

### Exercice 1 : le lac glissant, et ce que la stochasticité coûte

La section 5 a entraîné le Q-Learning sur le lac **glissant** avec $\varepsilon = 0,1$ fixe et 5 000 épisodes : la politique gloutonne atteint 81,8 %, et son écart à $V^*$ est de 0,023 au maximum. Reste à savoir ce que la stochasticité coûte **par rapport au lac déterministe**, à budget d'épisodes égal.

**À produire.** Le taux de réussite de la politique gloutonne sur `frozen` (déterministe) et sur `frozenSlippery`, tous deux évalués sur 2 000 épisodes sans exploration, avec le même nombre d'épisodes d'apprentissage.

**Indicateurs.**
- `QLearning(env, episodes, alpha, gamma, epsilon, seed)` renvoie `(Q, returns, steps, visits)`.
- `EvaluateGreedy(env, Q, episodes, seed)` mesure la politique sans exploration — c'est la colonne qui compte, pas la réussite en ligne.
- Le lac glissant a trois issues par action ; l'agent peut donc manquer le but depuis la case voisine, et c'est là que la différence se creusera.

In [24]:
// Exercice 1 : espace de travail
// TODO etudiant : entrainer le Q-Learning sur le lac DETERMINISTE et comparer les deux
//   politiques gloutonnes a budget d'episodes egal.
// Etape 1 : rejouer QLearning sur l'environnement `frozen` (5 000 episodes, Alpha, Gamma,
//           Epsilon, graine Seed) et conserver le Q obtenu.
// Etape 2 : evaluer les deux politiques gloutonnes sur 2 000 episodes, sans exploration.
// Indice : EvaluateGreedy(env, Q, episodes, graine) rend le taux de reussite ; la reussite
//   en ligne (returns) ne mesure pas la meme chose et ne doit pas servir ici.
double? rateDeterministic = null;   // TODO etudiant : remplacer par la mesure
double? rateSlippery = null;        // TODO etudiant : remplacer par la mesure

string Rate(double? v) => v.HasValue ? v.Value.ToString("P1") : "non mesure";

Console.WriteLine("Exercice 1 a completer");
Console.WriteLine($"  lac deterministe : {Rate(rateDeterministic)}");
Console.WriteLine($"  lac glissant     : {Rate(rateSlippery)}");
Console.WriteLine("  reference section 5 (glissant, 5 000 episodes, epsilon = 0.1) : 81.8 %");

Exercice 1 a completer


  lac deterministe : non mesure


  lac glissant     : non mesure


  reference section 5 (glissant, 5 000 episodes, epsilon = 0.1) : 81.8 %


### Exercice 2 : le taux d'apprentissage, et son optimum qui n'en est pas un

$\alpha$ arbitre entre deux erreurs : trop petit, la table avance à pas de fourmi et n'a pas le temps d'exploiter ses épisodes ; trop grand, chaque mise à jour écrase la précédente et la table oscille.

**À produire.** Le taux de réussite de la politique gloutonne pour $\alpha \in \{0{,}01\ ;\ 0{,}1\ ;\ 0{,}5\ ;\ 1{,}0\}$, tout le reste fixé (5 000 épisodes, $\gamma$, $\varepsilon$ et graine de la section 5).

**Indicateurs.**
- Une seule boucle sur les quatre valeurs ; `QLearning` puis `EvaluateGreedy` à chaque tour.
- La courbe attendue n'est pas monotone : il y a un optimum au milieu, et le lire vaut mieux que le connaître.
- $\alpha = 1$ n'apprend rien de stable — c'est un point de contrôle, pas une valeur à retenir.

In [25]:
// Exercice 2 : espace de travail
// TODO etudiant : balayer alpha sur {0.01, 0.1, 0.5, 1.0} et remplir le dictionnaire.
// Etape 1 : pour chaque alpha, appeler QLearning(frozenSlippery, 5000, alpha, Gamma, Epsilon, Seed).
// Etape 2 : mesurer EvaluateGreedy(frozenSlippery, Q, 2000, graine) et stocker le resultat.
// Indice : conservez les quatre Q si vous voulez aussi comparer les politiques, mais la
//   mesure demandee ne porte que sur le taux de reussite glouton.
var exercise2Rates = new Dictionary<double, double>();   // TODO etudiant : alpha -> reussite

Console.WriteLine($"Exercice 2 a completer : {exercise2Rates.Count} valeur(s) mesuree(s) sur 4");
foreach (var kv in exercise2Rates.OrderBy(k => k.Key))
    Console.WriteLine($"  alpha = {kv.Key,4:0.00} -> {kv.Value:P1}");

Exercice 2 a completer : 0 valeur(s) mesuree(s) sur 4


### Exercice 3 : la décroissance d'$\varepsilon$, et le régime que ce carnet n'a pas mesuré

Tout le carnet utilise la même décroissance, $\varepsilon \leftarrow \max(0{,}01\,;\ 0{,}999\,\varepsilon)$, qui atteint son plancher dès l'épisode ~4 600. C'est pourquoi le retour en ligne y suit le retour glouton : la politique mesurée à la fin n'explore presque plus. Le résultat classique — SARSA meilleur **pendant** l'apprentissage, Q-Learning meilleur **glouton** — demande un $\varepsilon$ qui reste haut.

**À produire.** Le retour glouton **et** le retour des 200 derniers épisodes pour trois calendriers sur CliffWalking : linéaire `max(0.01, 1 - episode/total)`, exponentiel (celui du carnet), et constant `0.1`.

**Indicateurs.**
- `TrainCliff(cliff, CliffRule.Sarsa, ...)` accepte déjà `epsStart`, `epsEnd`, `epsDecay` : le calendrier linéaire et le constant demandent une variante de la boucle, pas de la règle de mise à jour.
- C'est la colonne « en ligne » qui doit changer de camp, pas la colonne gloutonne.
- Attention au plancher : un $\varepsilon$ constant à 0,1 laisse 10 % de pas aléatoires jusqu'au dernier épisode, donc le retour en ligne ne peut pas rejoindre le glouton.

In [26]:
// Exercice 3 : espace de travail
// TODO etudiant : comparer trois calendriers de decroissance d'epsilon sur CliffWalking.
// Etape 1 : ecrire une variante d'entrainement qui prend une fonction episode -> epsilon
//           (les deux calendriers non exponentiels n'entrent pas dans les trois parametres
//           epsStart / epsEnd / epsDecay de TrainCliff).
// Etape 2 : pour chaque calendrier et chaque regle (Sarsa au minimum), mesurer
//           GreedyReturn(cliff, Q) et la moyenne des 200 derniers retours.
// Indice : le calendrier constant ne peut pas rejoindre le retour glouton — c'est justement
//   le point de l'exercice.
var exercise3Schedule = new Dictionary<string, (double Greedy, double Online)>();   // TODO etudiant

Console.WriteLine($"Exercice 3 a completer : {exercise3Schedule.Count} calendrier(s) mesure(s) sur 3");
foreach (var kv in exercise3Schedule)
    Console.WriteLine($"  {kv.Key,-12} : glouton {kv.Value.Greedy,6:0.0}   en ligne {kv.Value.Online,7:0.0}");

Exercice 3 a completer : 0 calendrier(s) mesure(s) sur 3


### Exercice 4 : $\alpha$ et $\gamma$ ensemble

Les deux hyperparamètres ne se règlent pas séparément : un $\gamma$ élevé rend le but lointain attractif, ce qui n'a de valeur que si $\alpha$ laisse la propagation remonter jusqu'au départ.

**À produire.** Une grille des taux de réussite gloutons pour $\alpha \in \{0{,}01\ ;\ 0{,}05\ ;\ 0{,}1\ ;\ 0{,}5\}$ croisé avec $\gamma \in \{0{,}9\ ;\ 0{,}95\ ;\ 0{,}99\ ;\ 1{,}0\}$, sur le lac glissant, 5 000 épisodes par couple.

**Indicateurs.**
- 16 entraînements, une matrice `[alpha, gamma]`, puis un affichage ligne par ligne.
- Le carnet n'a pas de bibliothèque de tracé matriciel : un tableau texte suffit, et `PlotSvg` reste disponible si vous voulez une figure.
- $\gamma = 1$ sur un MDP épisodique est légal ; c'est sur un MDP sans fin qu'il casse la convergence.

In [27]:
// Exercice 4 : espace de travail
// TODO etudiant : grille alpha x gamma sur le lac glissant, 5 000 episodes par couple.
// Etape 1 : definir les deux grilles de valeurs (4 valeurs chacune).
// Etape 2 : pour chaque couple, entrainer puis mesurer EvaluateGreedy sur 2 000 episodes.
// Etape 3 : afficher la matrice, une ligne par alpha, une colonne par gamma.
// Indice : 16 entrainements de 5 000 episodes restent rapides sur 16 etats ; inutile de
//   reduire le budget, ce serait changer la question.
double[] exercise4Alphas = Array.Empty<double>();   // TODO etudiant
double[] exercise4Gammas = Array.Empty<double>();   // TODO etudiant
double[,] exercise4Rates = null;                    // TODO etudiant : [alpha, gamma]

Console.WriteLine($"Exercice 4 a completer : grille {exercise4Alphas.Length} x {exercise4Gammas.Length}, matrice {(exercise4Rates == null ? "non remplie" : "remplie")}");

Exercice 4 a completer : grille 0 x 0, matrice non remplie


### Exercice 5 : le pont entre Q-Learning et SARSA

Les deux méthodes ne diffèrent que par une ligne — la cible. Cet exercice demande de construire le pont explicitement : une mise à jour qui commence **off-policy** (cible sur $\max_{a'} Q(s',a')$) pendant les 100 premiers épisodes, puis bascule **on-policy** (cible sur $Q(s', a'_{\text{jouée}})$).

**À produire.** La fonction hybride, puis son retour glouton sur les quatre graines du carnet, comparé à Q-Learning (−13,0) et SARSA (−17,0).

**Indicateurs.**
- La bascule est un simple changement de cible, pas un second algorithme : la table, le calendrier d'exploration et la graine restent les mêmes.
- Question à trancher par la mesure : les 100 premiers épisodes servent-ils de préchauffage (la seconde phase rattrape-t-elle la première ?) ou les deux phases laissent-elles deux régimes distincts dans la même table ?
- Un $\varepsilon$ encore élevé au moment de la bascule décide de la réponse : à l'épisode 100, il vaut environ $0{,}90$.

In [28]:
// Exercice 5 : espace de travail
// TODO etudiant : ecrire l'entrainement hybride, puis comparer son retour glouton.
// Signature attendue (a vous d'ecrire le corps) :
//   (double[,] Q, double[] Returns) TrainHybrid(CliffWorld env, int switchEpisode, int episodes,
//       double alpha, double gamma, double epsStart, double epsEnd, double epsDecay, int seed)
// Etape 1 : dans la boucle de TrainCliff, choisir la cible selon `ep < switchEpisode` :
//           max_a' Q(s', a') avant la bascule, Q(s', a' jouee) apres.
// Etape 2 : sur les quatre graines, mesurer GreedyReturn et comparer a -13.0 / -17.0.
// Indice : la section 7 a deja les deux cibles ecrites ; ici c'est le choix par episode
//   qui est nouveau, pas les formules.
double? exercise5Greedy = null;   // TODO etudiant : moyenne des 4 graines

string Show(double? v) => v.HasValue ? v.Value.ToString("0.0") : "non mesure";
Console.WriteLine("Exercice 5 a completer");
Console.WriteLine($"  hybride (100 episodes off-policy puis on-policy) : {Show(exercise5Greedy)}");
Console.WriteLine("  references : Q-Learning -13.0   |   SARSA -17.0");

Exercice 5 a completer


  hybride (100 episodes off-policy puis on-policy) : non mesure


  references : Q-Learning -13.0   |   SARSA -17.0


### Exercice 6 : la variance suit-elle l'horizon ?

La théorie annonce que $n \to \infty$ fait croître la variance : la cible cesse d'être un bootstrap et devient un retour complet, donc elle dépend de toute la trajectoire. La section 9 a mesuré $n \in \{1, 3, 5\}$ et n'a trouvé **aucune** dispersion inter-graines aux trois horizons — un résultat qui ne confirme pas la prédiction, et qu'il faut traiter comme tel.

**À produire.** L'écart-type du retour glouton sur quatre graines, pour $n \in \{1, 2, 3, 4, 5, 10\}$, et une conclusion honnête : soit la variance croît et où, soit elle ne croît pas et pourquoi.

**Indicateurs.**
- `TrainNStep(cliff, n, greedyBootstrap: false, ...)` accepte n'importe quel horizon ; seuls 1, 3 et 5 ont été mesurés.
- L'écart-type sur quatre points est lui-même très bruité : un écart nul à $n \leq 5$ puis non nul à $n = 10$ n'établit pas une tendance à lui seul.
- Si la variance ne bouge pas, cherchez du côté de la longueur des épisodes (17 pas en glouton) et du plancher de $\varepsilon$ : les deux raisons déjà nommées en section 9.

In [29]:
// Exercice 6 : espace de travail
// TODO etudiant : mesurer la dispersion inter-graines du retour glouton pour n in {1, 2, 3, 4, 5, 10}.
// Etape 1 : boucler sur les horizons, puis sur les quatre graines, avec TrainNStep(..., false, ...).
// Etape 2 : stocker moyenne, ecart-type et intervalle par horizon.
// Etape 3 : conclure explicitement — la variance croit, ou elle ne croit pas, et pourquoi.
// Indice : Std(v) existe deja (section 9) ; n = 10 depasse la longueur d'un episode glouton,
//   donc la cible n'est plus jamais un bootstrap sur les trajectoires courtes — c'est
//   precisement le cas ou la theorie predit l'ecart le plus grand.
var exercise6ByHorizon = new Dictionary<int, (double Mean, double Std, double Min, double Max)>();   // TODO etudiant

Console.WriteLine($"Exercice 6 a completer : {exercise6ByHorizon.Count} horizon(s) mesure(s) sur 6");
foreach (var kv in exercise6ByHorizon.OrderBy(k => k.Key))
    Console.WriteLine($"  n = {kv.Key,2} : moyenne {kv.Value.Mean,6:0.0}   std {kv.Value.Std,5:0.00}   [{kv.Value.Min,4:0} .. {kv.Value.Max,4:0}]");

Exercice 6 a completer : 0 horizon(s) mesure(s) sur 6


### Exercice 7 : SARSA($\lambda$) et les traces d'éligibilité

SARSA met à jour **une** paire état-action par pas : la dernière visitée. C'est un crédit différé d'un pas seulement. SARSA($\lambda$) répartit le crédit sur toutes les paires récemment visitées, pondérées par une trace $e_t(s,a)$ qui s'efface à chaque pas :

$$e_t(s,a) = \gamma\lambda\, e_{t-1}(s,a) + \mathbb{1}[S_t = s, A_t = a]$$

$$Q(s,a) \leftarrow Q(s,a) + \alpha \delta_t\, e_t(s,a), \qquad \delta_t = r + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)$$

**À produire.** L'implémentation, puis la comparaison de sa vitesse de convergence à SARSA(0) sur CliffWalking, à $\lambda \in \{0\ ;\ 0{,}5\ ;\ 0{,}9\}$.

**Indicateurs.**
- $\lambda = 0$ doit reproduire SARSA exactement : c'est le contrôle qui valide l'implémentation avant toute conclusion. La trace se réduit alors à l'indicatrice du dernier pas.
- La trace se remet à zéro au début de chaque épisode, sans exception : la laisser vivre d'un épisode à l'autre crédite un état visité dans une trajectoire où il n'apparaît pas.
- « Vitesse de convergence » se mesure : nombre d'épisodes pour atteindre un seuil de retour, ou retour moyen sur les 1 000 premiers épisodes. Retour glouton final et vitesse sont deux choses différentes, et $\lambda$ peut améliorer l'une sans l'autre.

**Référence de lecture.** Sutton et Barto, **§12.7 « Sarsa($\lambda$) »** (chapitre 12, *Eligibility Traces*) : la trace à valeurs d'action y est l'équation 12.23, et le pseudo-code complet de Sarsa($\lambda$) suit immédiatement — version à traces accumulantes ou remplaçantes, au choix.

In [30]:
// Exercice 7 : espace de travail
// TODO etudiant : implementer SARSA(lambda) avec traces d'eligibilite, puis le comparer a SARSA(0).
// Etape 1 : une table de traces z[etat, action] remise a zero au debut de CHAQUE episode.
// Etape 2 : a chaque pas, effacer toutes les traces par gamma * lambda, puis incrementer
//   celle de la paire jouee (traces accumulantes) — ou la fixer a 1 (traces remplacantes).
// Etape 3 : mettre a jour TOUTES les paires visitees avec alpha * delta * z, pas seulement la derniere.
// Etape 4 : verifier d'abord que lambda = 0 redonne SARSA exactement, PUIS conclure sur la vitesse.
// Indice : TrainCliff(env, CliffRule.Sarsa, episodes, alpha, gamma, epsStart, epsEnd, epsDecay, seed)
//   fournit la reference SARSA(0) ; le protocole de la section 7 (10 000 episodes, alpha 0.1,
//   gamma 0.99, epsilon 1.0 -> 0.01) est celui qui rend les deux mesures comparables.
var exercise7Lambdas = new[] { 0.0, 0.5, 0.9 };
var exercise7ByLambda = new Dictionary<double, (double Greedy, double Online)>();   // TODO etudiant

Console.WriteLine($"Exercice 7 a completer : {exercise7ByLambda.Count} valeur(s) de lambda mesuree(s) sur {exercise7Lambdas.Length}");
foreach (var kv in exercise7ByLambda.OrderBy(k => k.Key))
    Console.WriteLine($"  lambda = {kv.Key,3:0.0} : retour glouton {kv.Value.Greedy,6:0.0}   retour en ligne {kv.Value.Online,6:0.0}");

Exercice 7 a completer : 0 valeur(s) de lambda mesuree(s) sur 3


## 12. Synthèse : la cible de mise à jour est une posture de risque

Sur le même CliffWalking, aux mêmes hyperparamètres (10 000 épisodes, $\alpha = 0{,}1$, $\gamma = 0{,}99$, $\varepsilon$ de $1{,}0$ à $0{,}01$, quatre graines), les algorithmes ne rendent pas le même retour glouton. La seule chose qui les distingue est **la cible de leur mise à jour** — ce qu'ils écrivent dans $Q(s,a)$ comme valeur de la suite :

| Cible | Forme | Retour glouton | Ce qu'elle suppose |
|---|---|---|---|
| Maximum | $\max_{a'} Q(s',a')$ | $-13{,}0$ | l'agent jouera glouton à chaque pas |
| Espérance | $\mathbb{E}_\pi[Q(s',\cdot)]$ | $-15{,}0$ | l'agent se trompe, mais de façon moyenne |
| Échantillon | $Q(s',a')$ avec $a' \sim \pi$ | $-17{,}0$ | l'agent se trompe **de la façon dont il se trompe** |

Le maximum est optimiste : il évalue l'optimum *comme si* l'exploration n'existait pas, et sa trajectoire apprise longe la falaise d'un pas. L'échantillon paie chaque erreur d'exploration réelle : il évalue l'optimum d'un agent qui se trompe encore, et sa trajectoire prend du recul. L'espérance tient les deux par construction. Aucun n'est faux — ils répondent à des questions différentes, *que vaut le bord pour un agent parfait* et *que vaut le bord pour l'agent que je déploie*.

Les deux colonnes de mesures disent la même chose sous un autre angle. En **ligne** (moyenne des 200 derniers épisodes, donc l'agent qui explore encore), Q-Learning rend $-14{,}9$ et SARSA $-17{,}5$ : l'écart passe de $4{,}0$ à $2{,}6$ points, il ne se ferme pas. Un tableau qui ne montrerait que la colonne gloutonne exagérerait le prix de la prudence ; un tableau qui ne montrerait que la colonne en ligne le sous-estimerait.

**La frontière modèle / sans modèle est la seconde leçon.** Les quatre premières lignes du tableau de la section 10 ne portent aucune graine : itération de valeur et itération de politique lisent $P$ et n'interagissent jamais. Elles obtiennent $V^*(\text{départ}) = 0{,}951$ en 7 et 13 balayages, sur la même grille où Q-Learning plafonne à 81,8 % de réussite après 5 000 épisodes. Ce n'est pas que l'apprentissage soit « moins bon » : c'est qu'il ne dispose pas de la même information. Le jour où $P$ n'est pas disponible, seule la colonne de droite existe.

C'est la leçon à emporter avant d'approximer : le choix de la cible est un choix de déploiement, et il se fait en connaissance du régime dans lequel l'agent tournera.

## 13. Conclusion

| Concept | Ce que ce carnet en a fait, en C# |
|---|---|
| **MDP** | Formalisation $(S, A, P, R, \gamma)$ ; `GridEnvironment` porte `P[s,a]` comme liste de `Transition`, et deux grilles concrètes en dérivent (FrozenLake déterministe et glissant, CliffWalking) |
| **Itération de valeur** | Application répétée de l'équation de Bellman sur une table `double[16]` ; 7 balayages sur le lac déterministe, $\delta$ tombant **exactement** à zéro |
| **Itération de politique** | Alternance évaluation / amélioration ; 7 améliorations, 13 balayages d'évaluation, même politique que l'itération de valeur à l'arrivée |
| **Q-Learning** | Apprentissage sans modèle, cible $\max_{a'} Q(s',a')$ ; 81,8 % de réussite sur la politique gloutonne évaluée sans exploration, sans avoir jamais lu $P$ |
| **FrozenLake** | Grille discrète à deux régimes ; la stochasticité ne change pas la politique optimale, elle change le coût pour y arriver (7 balayages contre 324) |
| **CliffWalking** | Le compromis risque / récompense, et la différence **mesurée** entre cibles de mise à jour : $-13$ / $-15$ / $-17$ |
| **n-step TD** | Une seule boucle, deux règles d'amorçage ; $n = 1$ recoupe exactement SARSA ($-17{,}0$) et Q-Learning ($-13{,}0$), ce qui valide l'implémentation des deux côtés |

**Ce que le C# a demandé, et qui ne se voit pas dans le résultat.** Le noyau garde son état d'une cellule à l'autre : les fonctions locales et les `sealed class` déclarées dans une cellule sont visibles dans les suivantes, ce qui permet de découper le carnet en sections autonomes sans rien réimporter. Deux conséquences pratiques : une cellule doit être exécutée dans l'ordre pour que les types existent, et une figure tracée en SVG doit passer par un `Display()` explicite — la valeur implicite d'une expression n'est pas persistée par le canal d'exécution, alors qu'un `display_data` l'est.

**Prochaines étapes.** Le carnet suivant de la série aborde l'approximation de fonction : quand la table devient impossible à tenir (états continus, ou simplement trop nombreux), $Q$ devient un réseau et les garanties de convergence tabulaire tombent avec la table. Le jumeau Python de ce carnet, [RL-05-MDP-Programmation-Dynamique-Q-Learning-Python.ipynb](RL-05-MDP-Programmation-Dynamique-Q-Learning-Python.ipynb), suit le même plan ; les deux se lisent côte à côte, et leurs chiffres se comparent — ils ne coïncident pas toujours, et la section 9 dit où et pourquoi.

## 14. À retenir

- **Deux routes, un même point fixe.** Sur le lac déterministe (16 états, 4 actions), l'itération de valeur converge en **7 balayages** avec $\delta$ tombant **exactement à zéro**, et l'itération de politique en **7 améliorations pour 13 balayages d'évaluation** — même politique à l'arrivée, écart $|V_\pi - V^*| = 0$. Sur le lac glissant, les deux comptes deviennent **324** et **976**. Le nombre d'itérations n'est pas une propriété de l'algorithme : il est une propriété du MDP.

- **L'itération de politique coûte plus cher que l'itération de valeur, ici, sur les deux lacs** (13 contre 7, 976 contre 324). Ce n'est pas une propriété de l'algorithme non plus : avec 16 et 48 états, chaque amélioration doit être payée par une évaluation menée presque à convergence, et les deux méthodes font le même nombre de cycles (7). L'avantage classique de l'itération de politique porte sur le **nombre de cycles**, pas sur le nombre de balayages — il faut un MDP assez grand pour que ce soit lui qui compte.

- **Q-Learning tabulaire apprend sans modèle, et cela se mesure.** Sur 5 000 épisodes, sa politique gloutonne évaluée **sans exploration** réussit **81,8 %** du temps, avec $V(\text{départ}) = 0{,}5362$ contre $V^* = 0{,}5420$ — écart maximal $0{,}0231$ sur les 11 états non terminaux, et la même politique gloutonne sur ces 11 états. $P$ était disponible depuis la section 2 et n'a jamais été lu par cet agent.

- **Les balayages mesurent, ils n'illustrent pas — et la moyenne cache la distribution.** $\alpha$ passe de $15{,}7\ \%$ ($0{,}01$) à $81{,}3\ \%$ ($0{,}10$), puis redescend à $18{,}6\ \%$ ($0{,}90$) : un optimum intérieur, pas un bord. Mais à $\alpha = 0{,}01$ la moyenne de $15{,}7\ \%$ recouvre quatre graines à $6{,}2$, $2{,}2$, $2{,}8$ et **$51{,}4\ \%$** — une graine sur quatre apprend, les trois autres non. Un chiffre moyen sans sa dispersion aurait conclu « $\alpha$ petit n'apprend pas », alors que la bonne conclusion est « $\alpha$ petit apprend parfois, et le résultat dépend de la graine ».

- **$\gamma$ élevé sans budget plus grand achète une égalité, pas une connaissance.** À 5 000 épisodes, $\gamma = 0{,}99$ donne $81{,}3\ \%$ et $\gamma = 0{,}999$ retombe à $60{,}6\ \%$. Sur la mesure voisine (graine 42), $\gamma = 0{,}999$ donne $0{,}0\ \%$ avec les quatre actions de l'état 0 dans un intervalle de $5{,}3\times10^{-4}$ — la politique gloutonne y est un tirage à pile ou face entre quatre actions équivalentes. L'actualisation relie une action lointaine à sa conséquence ; elle ne crée pas le signal qui manque.

- **Le budget d'exploration n'est pas un paramètre de confort.** Sur la graine 0, à $\varepsilon = 0{,}05$, deux états sur onze n'essayent jamais leur action optimale, et le nombre d'essais va de **11 à 39 758** selon l'état — un facteur 3 614 qui explique pourquoi « $\varepsilon$ petit » n'est pas « $\varepsilon$ suffisant ». À $\varepsilon = 0{,}20$, un seul état reste faux sur onze.

- **CliffWalking : la cible de mise à jour est une posture de risque.** Mêmes hyperparamètres, 48 cases : Q-Learning $-13{,}0$ (le chemin optimal, à un pas de la falaise), Expected SARSA $-15{,}0$, SARSA $-17{,}0$ (le chemin qui s'en écarte). Les trois ne cherchent pas la même chose — le meilleur chiffre n'est pas la meilleure politique pour un agent qui explore encore.

- **L'horizon n-step ne fait pas ce qu'on croit.** À $n = 1$, $3$ et $5$ avec l'amorçage sur l'action jouée, le retour glouton reste $-17{,}0$ : ce qui déplace la colonne, c'est la **règle d'amorçage** (gloutonne : $-13{,}0$ dès $n = 1$), pas la longueur de l'horizon. Et le compromis biais-variance n'est pas visible ici : la dispersion inter-graines vaut $0{,}00$ partout, donc ce protocole ne peut pas le montrer.

## 15. Vérifiez votre compréhension

**1. L'itération de valeur converge en 7 balayages et l'itération de politique en 13. Peut-on dire que la première est deux fois plus rapide ?**

<details><summary>Réponse</summary>

Non — ce sont deux unités différentes. L'itération de valeur compte des **balayages de Bellman** sur l'espace d'états entier ; l'itération de politique compte des **balayages d'évaluation** répartis dans 7 cycles d'amélioration. Ce qui reste comparable est le coût total, et il vaut 7 contre 13 sur le lac déterministe, 324 contre 976 sur le lac glissant : l'itération de valeur est moins chère sur les deux.

Ce qui ne se transpose pas, c'est le **nombre de cycles** : il est identique ici (7 et 7). L'avantage classique de l'itération de politique porte justement sur ce nombre-là — elle en fait moins — et il faut un MDP assez grand pour que ce soit lui qui décide du coût.

</details>

**2. Q-Learning rend $-13{,}0$ et SARSA $-17{,}0$. Peut-on conclure que Q-Learning est meilleur ?**

<details><summary>Réponse</summary>

Non. Q-Learning apprend la politique **gloutonne optimale**, qui longe la falaise d'un pas : excellente si l'agent joue glouton, coûteuse dès qu'il explore encore. SARSA évalue la politique **$\varepsilon$-gloutonne qu'il suit réellement**, donc il apprend un chemin qui s'écarte de la falaise.

La colonne en ligne mesure exactement cela : un agent qui explore encore voit $-14{,}9$ pour Q-Learning et $-17{,}5$ pour SARSA. L'écart tombe de $4{,}0$ à $2{,}6$ points — il se réduit, mais il ne se ferme pas. Le bon chiffre dépend donc du régime de déploiement, et c'est la question à se poser avant de choisir.

</details>

**3. L'ablation n-step rend $-17{,}0$ pour $n = 1$, $3$ et $5$. L'horizon est-il sans effet ?**

<details><summary>Réponse</summary>

Non, et l'ablation elle-même le montre par deux fois. D'abord la **règle d'amorçage** déplace la colonne : à $n = 1$ avec l'amorçage glouton (hors politique), on retrouve exactement Q-Learning à $-13{,}0$ — ce n'est donc pas l'horizon qui décide mais ce qu'on met à la place de la fin de l'épisode. Ensuite la moyenne des 200 premiers épisodes bouge encore avec $n$ : $-17{,}5$, $-17{,}3$, $-17{,}3$.

Ce que cette ablation **ne** montre pas, c'est le compromis biais-variance : le retour glouton est saturé et la dispersion inter-graines vaut $0{,}00$ aux trois horizons. Un résultat nul sur une mesure saturée n'est pas une preuve d'absence d'effet — c'est un protocole muet, et la section 9 le dit plutôt que de conclure à sa place.

</details>

## 16. Pour aller plus loin

- **[RL-08 — Dyna-Q et planification](RL-08-Dyna-Q-Planification-CSharp.ipynb)** : le pont direct entre les deux moitiés de ce carnet. L'agent y apprend un modèle à partir de ses transitions, puis **planifie** avec — ce qui revient à réinjecter l'itération de valeur de la section 3 dans un agent qui n'a pas $P$ au départ.
- **[RL-06b — Actor-Critic](RL-06b-Actor-Critic-CSharp.ipynb)** : quand la table ne tient plus. $Q$ devient une fonction approchée, et les garanties de convergence tabulaire de ce carnet tombent avec la table — c'est le prix, et le carnet le montre.
- **[RL-11 — POMDP et croyances](RL-11-POMDP-Croyances-CSharp.ipynb)** : ce qui casse quand l'agent n'observe plus l'état mais une observation bruitée, et pourquoi une table sur les états ne suffit alors plus.
- **[RL-17 — Reward Tampering](RL-17-Reward-Tampering-CSharp.ipynb)** : ce qui arrive quand la récompense elle-même est apprise ou manipulable. La cible de mise à jour de la section 12 suppose une récompense honnête ; ce carnet retire l'hypothèse.
- **[Sutton & Barto, *Reinforcement Learning: An Introduction*, 2ᵉ éd.](http://incompleteideas.net/book/the-book.html)** — chapitres 3 à 4 (MDP, programmation dynamique), 6 (TD et SARSA), 7 (n-step), 12 (traces d'éligibilité). Le chapitre 12 est celui que l'exercice 7 demande de mettre en pratique.
- **Le jumeau Python de ce carnet**, [RL-05-MDP-Programmation-Dynamique-Q-Learning-Python.ipynb](RL-05-MDP-Programmation-Dynamique-Q-Learning-Python.ipynb), pour comparer deux implémentations du même protocole — et voir où elles divergent, ce que la section 9 documente.